## accounts app ##


# ────────────────────────Start URL─────────────────────────────────────

```
"""
Accounts API URL configuration.
Maps all account-related endpoints to their views.

Endpoints:
    POST   /api/auth/register/          → RegisterView
    POST   /api/auth/login/             → LoginView
    POST   /api/auth/token/refresh/     → TokenRefreshView (built-in simplejwt)
    GET    /api/profile/                → ProfileView
    PATCH  /api/profile/                → ProfileView
    GET    /api/profile/<id>/           → PublicProfileView
    POST   /api/verify/documents/       → VerificationDocumentView
    GET    /api/verify/documents/       → VerificationDocumentView
"""

from django.urls import path
from rest_framework_simplejwt.views import TokenRefreshView
from api.accounts_api.views import (
    RegisterView,
    LoginView,
    ProfileView,
    PublicProfileView,
    VerificationDocumentView,
)

urlpatterns = [
    # ── Auth ──────────────────────────────────────────────────
    path("register/", RegisterView.as_view(), name="register"),
    path("login/", LoginView.as_view(), name="login"),
    path("token/refresh/", TokenRefreshView.as_view(), name="token-refresh"),  # built-in

    # ── Profile ───────────────────────────────────────────────
    path("profile/", ProfileView.as_view(), name="my-profile"),
    path("profile/<int:user_id>/", PublicProfileView.as_view(), name="public-profile"),

    # ── Verification ──────────────────────────────────────────
    path("verify/documents/", VerificationDocumentView.as_view(), name="verify-documents"),
]

        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```
"""
Accounts API serializers.
Handles serialization for auth, profiles, and verification.

Serializers:
    - UserSerializer                → read user data (used inside other serializers)
    - RegisterSerializer            → POST /api/auth/register/
    - LoginSerializer               → POST /api/auth/login/
    - TokenSerializer               → login response shape (tokens + user)
    - UserUpdateSerializer          → PATCH /api/profile/
    - StudentProfileSerializer      → GET/PATCH student profile section
    - LandlordProfileSerializer     → GET/PATCH landlord profile section
    - VerificationDocumentSerializer → POST /api/verify/documents/
"""

from rest_framework import serializers
from accounts.models import Users, StudentProfile, LandlordProfile, VerificationDocument
from accounts.services import create_user_account, authenticated
from accounts.validators import validate_phone_number

# ──────────────────────────────────────────────────────────────────────────────────────────


class StudentProfileSerializer(serializers.ModelSerializer):
    """
    Serializes the StudentProfile model.
    Nested inside UserSerializer so the profile page gets everything in one request.
    Also used standalone for PATCH /api/profile/ (student section only).
    """

    class Meta:
        model = StudentProfile
        fields = [
            # ── About Me ──────────────────────────────────────
            "bio",
            "university",
            "faculty",
            "year_of_study",
            "languages",
            "profile_strength",
            "is_looking_for_room",
            # ── Roommate Matching ─────────────────────────────
            "lifestyle_tags",
            # ── Interests & Lifestyle ─────────────────────────
            "sleeping_time",
            "study_environment",
            "music_preference",
            "guests_policy",
            "personality",
            "cleanliness",
            "smoking",
            "budget_min",
            "budget_max",
            "room_type_preference",
            # ── Personal Preferences ──────────────────────────
            "preferred_room_type",
            "smoking_preference",
            "sleep_schedule_pref",
            "cleanliness_pref",
            "personality_pref",
        ]


# ──────────────────────────────────────────────────────────────────────────────────────────


class LandlordProfileSerializer(serializers.ModelSerializer):
    """
    Serializes the LandlordProfile model.
    Nested inside UserSerializer for landlord profile page.
    Also used standalone for PATCH /api/profile/ (landlord section only).

    NOTE: total_income and available_balance are read-only —
          they get updated by the payments app, not by the user.
    """

    class Meta:
        model = LandlordProfile
        fields = [
            "company_name",
            "national_id",
            "is_id_verified",   # read-only — admin sets this
            "total_income",     # read-only — calculated from payments
            "available_balance", # read-only — calculated from payments
        ]
        read_only_fields = ["is_id_verified", "total_income", "available_balance"]


# ──────────────────────────────────────────────────────────────────────────────────────────


class UserSerializer(serializers.ModelSerializer):
    """
    Full user profile serializer.
    Used for GET /api/profile/ and GET /api/profile/<id>/

    Nests the correct profile based on role:
        role = student  → student_profile is populated, landlord_profile is null
        role = landlord → landlord_profile is populated, student_profile is null
    """

    # Nested profiles — read only, updated via their own serializers
    student_profile = StudentProfileSerializer(read_only=True)
    landlord_profile = LandlordProfileSerializer(read_only=True)

    class Meta:
        model = Users
        fields = [
            # ── Identity ──────────────────────────────────────
            "id",
            "username",
            "email",
            "role",             
            # ── Personal Info ─────────────────────────────────
            "first_name",
            "last_name",
            "phone_number",
            "gender",
            "date_of_birth",
            "profile_picture",
            "city",
            # ── Profile Badges ────────────────────────────────
            "is_verified",          
            "is_top_rated",         
            "is_quick_responder",   
            # ── Nested Profiles ───────────────────────────────
            "student_profile",
            "landlord_profile",
        ]
        read_only_fields = ["id", "role", "is_verified", "is_top_rated", "is_quick_responder"]


# ──────────────────────────────────────────────────────────────────────────────────────────


class RegisterSerializer(serializers.ModelSerializer):
    """
    Handles new user registration.
    Used for POST /api/auth/register/

    IMPORTANT: `role` must be included — it triggers the signal that
    auto-creates either StudentProfile or LandlordProfile.
    """

    password = serializers.CharField(write_only=True, min_length=8)

    class Meta:
        model = Users
        fields = [
            "username",
            "email",
            "first_name",
            "last_name",
            "password",
            "phone_number",
            "gender",
            "date_of_birth",
            "profile_picture",
            "role",             # required — determines which profile gets created
        ]
        extra_kwargs = {
            "email": {"required": True},
        }

    def validate_phone_number(self, value):
        """Delegate phone validation to the shared validator."""
        return validate_phone_number(value)

    def validate_role(self, value):
        """Only allow valid role values — reject anything unexpected."""
        if value not in ["student", "landlord"]:
            raise serializers.ValidationError("Role must be 'student' or 'landlord'.")
        return value

    def create(self, validated_data):
        """Pop password and delegate user creation to the service layer."""
        password = validated_data.pop("password")
        user = create_user_account(password=password, **validated_data)
        return user


# ──────────────────────────────────────────────────────────────────────────────────────────


class LoginSerializer(serializers.Serializer):
    """
    Validates login credentials.
    Used for POST /api/auth/login/

    Supports login with username or email.
    """

    username = serializers.CharField()  # accepts username or email
    password = serializers.CharField(write_only=True)  # never returned in response

    def validate(self, data):
        """Authenticate user and return user object, or raise error."""
        identifier = data["username"].strip()
        password = data["password"]

        # First try direct username authentication.
        user = authenticated(username=identifier, password=password)

        # Fallback: allow email in the same field.
        if not user and "@" in identifier:
            matched_user = Users.objects.filter(email__iexact=identifier).first()
            if matched_user:
                user = authenticated(username=matched_user.username, password=password)

        if not user:
            raise serializers.ValidationError("Invalid username or password.")

        return {"user": user}


# ──────────────────────────────────────────────────────────────────────────────────────────


class TokenSerializer(serializers.Serializer):
    """
    Shape of the login response.
    Returns both JWT tokens + full user object so the frontend
    can store everything it needs in one request.
    """

    access = serializers.CharField()
    refresh = serializers.CharField()
    user = UserSerializer()


# ──────────────────────────────────────────────────────────────────────────────────────────


class UserUpdateSerializer(serializers.ModelSerializer):
    """
    Handles partial profile updates.
    Used for PATCH /api/profile/

    Splits into 2 steps:
        1. Update top-level Users fields (name, phone, etc.)
        2. Update nested profile fields (StudentProfile or LandlordProfile)
    """

    # Accept nested profile data in the same PATCH request
    # Both are optional — only the relevant one will be used based on role
    student_profile = StudentProfileSerializer(required=False)
    landlord_profile = LandlordProfileSerializer(required=False)

    class Meta:
        model = Users
        fields = [
            "email",
            "first_name",
            "last_name",
            "phone_number",
            "gender",
            "date_of_birth",
            "profile_picture",
            "city",
            "student_profile",
            "landlord_profile",
        ]

    def validate_phone_number(self, value):
        """Delegate phone validation to the shared validator."""
        return validate_phone_number(value)

    def update(self, instance, validated_data):
        """
        Update Users fields first, then update the nested profile if provided.
        Uses pop() so nested data doesn't get passed to the Users model update.
        """
        # ── Extract nested profile data before updating Users ──
        student_data = validated_data.pop("student_profile", None)
        landlord_data = validated_data.pop("landlord_profile", None)

        # ── Update top-level Users fields ─────────────────────
        for field, value in validated_data.items():
            setattr(instance, field, value)
        instance.save()

        # ── Update StudentProfile if user is a student ─────────
        if student_data and instance.is_student:
            student_profile = instance.student_profile
            for field, value in student_data.items():
                setattr(student_profile, field, value)
            student_profile.save()

        # ── Update LandlordProfile if user is a landlord ───────
        if landlord_data and instance.is_landlord:
            landlord_profile = instance.landlord_profile
            for field, value in landlord_data.items():
                setattr(landlord_profile, field, value)
            landlord_profile.save()

        return instance


# ──────────────────────────────────────────────────────────────────────────────────────────


class VerificationDocumentSerializer(serializers.ModelSerializer):
    """
    Handles document uploads for student verification.
    Used for POST /api/verify/documents/

    Accepted types: national_id, student_id, university_email
    Admin reviews uploads and sets is_verified = True manually.
    """

    class Meta:
        model = VerificationDocument
        fields = [
            "id",
            "doc_type",
            "file",
            "is_verified", 
            "uploaded_at",
        ]
        read_only_fields = ["id", "is_verified", "uploaded_at"]

```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```
"""
Accounts API views.
Handles all request/response logic for auth and profiles.

Views:
    - RegisterView          → POST /api/auth/register/
    - LoginView             → POST /api/auth/login/
    - ProfileView           → GET/PATCH /api/profile/
    - PublicProfileView     → GET /api/profile/<id>/
    - VerificationDocumentView → POST /api/verify/documents/
"""

from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework.permissions import IsAuthenticated, AllowAny
from rest_framework import status
from rest_framework_simplejwt.tokens import RefreshToken
from accounts.models import Users, VerificationDocument
from api.accounts_api.serializers import (
    RegisterSerializer,
    LoginSerializer,
    TokenSerializer,
    UserSerializer,
    UserUpdateSerializer,
    VerificationDocumentSerializer,
)


# ── Helpers ───────────────────────────────────────────────────────────────────────────────


def get_tokens_for_user(user):
    """
    Generates a JWT access + refresh token pair for a given user.
    Called after register and login.
    """
    refresh = RefreshToken.for_user(user)
    return {
        "refresh": str(refresh),
        "access": str(refresh.access_token),
    }


# ──────────────────────────────────────────────────────────────────────────────────────────


class RegisterView(APIView):
    """
    POST /api/auth/register/
    Creates a new user and returns JWT tokens + user data.
    No authentication required.
    """

    permission_classes = [AllowAny]

    def post(self, request):
        """Validate registration data, create user, return tokens."""
        serializer = RegisterSerializer(data=request.data)

        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        user = serializer.save()
        tokens = get_tokens_for_user(user)

        response_data = TokenSerializer({
            **tokens,
            "user": user,
        }).data

        return Response(response_data, status=status.HTTP_201_CREATED)


# ──────────────────────────────────────────────────────────────────────────────────────────


class LoginView(APIView):
    """
    POST /api/auth/login/
    Validates credentials and returns JWT tokens + user data.
    No authentication required.
    """

    permission_classes = [AllowAny]

    def post(self, request):
        """Validate credentials, return tokens if correct."""
        serializer = LoginSerializer(data=request.data)

        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        user = serializer.validated_data["user"]
        tokens = get_tokens_for_user(user)

        response_data = TokenSerializer({
            **tokens,
            "user": user,
        }).data

        return Response(response_data, status=status.HTTP_200_OK)


# ──────────────────────────────────────────────────────────────────────────────────────────


class ProfileView(APIView):
    """
    GET  /api/profile/ → returns the logged-in user's full profile
    PATCH /api/profile/ → updates the logged-in user's profile

    Must be authenticated. Users can only see/edit their own profile here.
    To view someone else's profile use PublicProfileView.
    """

    permission_classes = [IsAuthenticated]

    def get(self, request):
        """Return full profile of the currently logged-in user."""
        serializer = UserSerializer(request.user)
        return Response(serializer.data, status=status.HTTP_200_OK)

    def patch(self, request):
        """Partially update the logged-in user's profile."""
        serializer = UserUpdateSerializer(
            request.user,
            data=request.data,
            partial=True,  # allow updating only some fields
        )

        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        serializer.save()
        # Return full updated profile using UserSerializer
        return Response(UserSerializer(request.user).data, status=status.HTTP_200_OK)


# ──────────────────────────────────────────────────────────────────────────────────────────


class PublicProfileView(APIView):
    """
    GET /api/profile/<id>/
    View another user's public profile.
    Used when clicking on a student or landlord profile card.
    Must be authenticated to view.
    """

    permission_classes = [IsAuthenticated]

    def get(self, request, user_id):
        """Return public profile of any user by ID."""
        try:
            user = Users.objects.get(id=user_id)
        except Users.DoesNotExist:
            return Response(
                {"error": "User not found."},
                status=status.HTTP_404_NOT_FOUND
            )

        serializer = UserSerializer(user)
        return Response(serializer.data, status=status.HTTP_200_OK)


# ──────────────────────────────────────────────────────────────────────────────────────────


class VerificationDocumentView(APIView):
    """
    POST /api/verify/documents/
    Student uploads a verification document (National ID, Student ID, etc.)
    Admin reviews it manually and sets is_verified = True.
    Must be authenticated.
    """

    permission_classes = [IsAuthenticated]

    def post(self, request):
        """Upload a new verification document."""
        serializer = VerificationDocumentSerializer(data=request.data)

        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        # Attach the document to the logged-in user
        serializer.save(user=request.user)
        return Response(serializer.data, status=status.HTTP_201_CREATED)

    def get(self, request):
        """Return all documents uploaded by the logged-in user."""
        documents = VerificationDocument.objects.filter(user=request.user)
        serializer = VerificationDocumentSerializer(documents, many=True)
        return Response(serializer.data, status=status.HTTP_200_OK)
```

# ────────────────────────End View──────────────────────────────────────




# ────────────────────────Start permissions────────────────────────────────────
```
"""
Accounts API custom permissions.
Used as permission_classes in views instead of repeating role checks everywhere.

Permissions:
    - IsStudent  → only students can access
    - IsLandlord → only landlords can access
"""

from rest_framework.permissions import BasePermission


# ──────────────────────────────────────────────────────────────────────────────────────────


class IsStudent(BasePermission):
    """
    Allows access only to users with role = 'student'.
    Used on: roommate matching, shortlist, student profile endpoints.
    """

    message = "Access restricted to students only."

    def has_permission(self, request, view):
        """Return True only if the user is authenticated and is a student."""
        return bool(
            request.user and
            request.user.is_authenticated and
            request.user.role == "student"
        )


class IsLandlord(BasePermission):
    """
    Allows access only to users with role = 'landlord'.
    Used on: create property, owner dashboard, payments endpoints.
    """

    message = "Access restricted to landlords only."

    def has_permission(self, request, view):
        """Return True only if the user is authenticated and is a landlord."""
        return bool(
            request.user and
            request.user.is_authenticated and
            request.user.role == "landlord"
        )

```
# ────────────────────────End permissions──────────────────────────────────────






# ────────────────────────Start Signals────────────────────────────────────
```
"""
Accounts app signals.
Auto-creates the correct profile when a new user registers.

Signals:
    - create_user_profile → fires after Users is saved, creates StudentProfile or LandlordProfile
"""

from django.db.models.signals import post_save
from django.dispatch import receiver
from accounts.models import Users, StudentProfile, LandlordProfile


# ──────────────────────────────────────────────────────────────────────────────────────────


@receiver(post_save, sender=Users)
def create_user_profile(sender, instance, created, **kwargs):
    """
    Fires every time a Users object is saved.
    Only runs on creation (created=True) to avoid duplicate profiles.

    role = student  → creates StudentProfile
    role = landlord → creates LandlordProfile
    """
    if not created:
        return  # user is being updated, not created — do nothing

    if instance.role == "student":
        StudentProfile.objects.create(user=instance)

    elif instance.role == "landlord":
        LandlordProfile.objects.create(user=instance)

```
# ────────────────────────End Signals──────────────────────────────────────





# ────────────────────────Start Apps────────────────────────────────────
```
"""
Accounts app configuration.
The ready() method imports signals so they register when Django starts.
Without this, signals.py exists but never fires.
"""

from django.apps import AppConfig


class AccountsConfig(AppConfig):
    """Configuration class for the accounts app."""

    default_auto_field = "django.db.models.BigAutoField"
    name = "accounts"

    def ready(self):
        """Import signals when the app is ready so they connect to the dispatcher."""
        import accounts.signals  # noqa: F401

```
# ────────────────────────End Apps──────────────────────────────────────



# ────────────────────────Start Validators────────────────────────────────────
```

"""
Accounts app validators.
Reusable validation functions imported by serializers.

Validators:
    - validate_phone_number → ensures phone is in valid international format
"""

from rest_framework import serializers


# ──────────────────────────────────────────────────────────────────────────────────────────


def validate_phone_number(value):
    """
    Validates that the phone number is not empty.
    The PhoneNumberField on the model handles format validation automatically.
    This function is a hook for any extra rules we want to enforce.
    """
    if value and str(value).strip() == "":
        raise serializers.ValidationError("Phone number cannot be blank.")
    return value
```
# ────────────────────────End Validators──────────────────────────────────────



# ────────────────────────Start Services────────────────────────────────────
```
"""
Accounts app services.
Business logic layer — keeps views and serializers clean.

Functions:
    - create_user_account → creates a new user with hashed password
    - authenticated       → verifies credentials and returns user or None
"""

from django.contrib.auth import authenticate
from accounts.models import Users


# ──────────────────────────────────────────────────────────────────────────────────────────


def create_user_account(password, **kwargs):
    """
    Creates a new user with a properly hashed password.

    Using create_user() instead of create() is critical —
    create() stores the password in plain text which breaks authentication.

    Called by: RegisterSerializer.create()
    """
    user = Users.objects.create_user(password=password, **kwargs)
    return user


def authenticated(username, password):
    """
    Checks credentials and returns the user object if valid, otherwise None.

    Using Django's built-in authenticate() so it respects
    any auth backends configured in settings.

    Called by: LoginSerializer.validate()
    """
    user = authenticate(username=username, password=password)
    return user  # None if credentials are wrong

```
# ────────────────────────End Services──────────────────────────────────────






# ────────────────────────Start Model────────────────────────────────────

```
"""
Accounts app models.
Handles all user authentication, profiles, and verification.

Models:
    - Users                → base user for both students and landlords
    - StudentProfile       → lifestyle and preferences (students only)
    - LandlordProfile      → business and financial info (landlords only)
    - VerificationDocument → uploaded IDs and documents for verification
"""

from django.db import models
from django.contrib.auth.models import AbstractUser
from phonenumber_field.modelfields import PhoneNumberField

# ──────────────────────────────────────────────────────────────────────────────────────────


class Users(AbstractUser):
    """
    Main user model for the entire app.
    Handles both Students and Landlords via the `role` field.

    Student gets → StudentProfile (lifestyle, preferences)
    Landlord gets → LandlordProfile (properties, payments)
    """

    # ── Role ────────────────────────────────────────────────
    ROLE_CHOICES = [("student", "Student"), ("landlord", "Landlord")]
    role = models.CharField(max_length=10, choices=ROLE_CHOICES, default="student")

    # ── Personal Info (shared by both roles) ────────────────
    GENDER_CHOICES = [("M", "Male"), ("F", "Female")]
    first_name = models.CharField(max_length=150, blank=True, null=True)
    last_name = models.CharField(max_length=150, blank=True, null=True)
    phone_number = PhoneNumberField(blank=True, null=True)
    date_of_birth = models.DateField(blank=True, null=True)
    gender = models.CharField(
        max_length=1, choices=GENDER_CHOICES, blank=True, null=True
    )
    profile_picture = models.ImageField(
        upload_to="profile_pics/", blank=True, null=True
    )
    city = models.CharField(max_length=100, blank=True, null=True)

    # ── Profile Badges (shown on profile page) ──────────────
    is_verified = models.BooleanField(default=False)  #  Verified badge
    is_top_rated = models.BooleanField(default=False)  #  Top Rated badge
    is_quick_responder = models.BooleanField(default=False)  #  Quick Responder badge

    def __str__(self):
        return self.username

    # ── Helper Properties (use in views/serializers) ────────
    @property
    def is_student(self):
        """Returns True if this user is a student."""
        return self.role == "student"

    @property
    def is_landlord(self):
        """Returns True if this user is a landlord."""
        return self.role == "landlord"


# ──────────────────────────────────────────────────────────────────────────────────────────


class StudentProfile(models.Model):
    """
    Extra fields for students only.
    Auto-created by signal when a student registers.

    Sections:
    - About Me (bio, university, faculty)
    - Interests & Lifestyle (sleeping, personality, smoking...)
    - Personal Preferences (what they want in a roommate)

    Used in: Profile page, Roommate Matching page
    """

    # ── Choices ─────────────────────────────────────────────
    SLEEPING_CHOICES = [
        ("early", "Early (9-10 PM)"),
        ("normal", "Normal (11 PM)"),
        ("late", "Late (12+ AM)"),
    ]
    CLEANLINESS_CHOICES = [("low", "Low"), ("medium", "Medium"), ("high", "High")]
    PERSONALITY_CHOICES = [
        ("quiet", "Quiet"),
        ("social", "Social"),
        ("moderate", "Moderate"),
    ]
    SMOKING_CHOICES = [("smoker", "Smoker"), ("non_smoker", "Non-Smoker")]
    GUESTS_CHOICES = [
        ("never", "Never"),
        ("sometimes", "Sometimes"),
        ("often", "Often"),
    ]
    ROOM_TYPE_CHOICES = [("single", "Single"), ("shared", "Shared"), ("both", "Both")]

    # ── Relation ─────────────────────────────────────────────
    # OneToOne means one student = one profile, deleting user deletes this too
    user = models.OneToOneField(
        Users, on_delete=models.CASCADE, related_name="student_profile"
    )

    # ── About Me Section ─────────────────────────────────────
    bio = models.TextField(blank=True, null=True)
    university = models.CharField(max_length=255, blank=True, null=True)
    faculty = models.CharField(max_length=255, blank=True, null=True)
    year_of_study = models.CharField(max_length=50, blank=True, null=True)
    languages = models.JSONField(default=list, blank=True)
    profile_strength = models.IntegerField(default=0)  # 0-100, shown as % on profile
    is_looking_for_room = models.BooleanField(default=False)

    # ── Roommate Matching ─────────────────────────────────────
    # Tags shown on roommate cards e.g. ["Quiet", "Clean", "Night owl"]
    lifestyle_tags = models.JSONField(default=list, blank=True)

    # ── Interests & Lifestyle Section ────────────────────────
    sleeping_time = models.CharField(
        max_length=20, choices=SLEEPING_CHOICES, blank=True, null=True
    )
    study_environment = models.CharField(max_length=50, blank=True, null=True)
    music_preference = models.CharField(max_length=50, blank=True, null=True)
    guests_policy = models.CharField(
        max_length=20, choices=GUESTS_CHOICES, blank=True, null=True
    )
    personality = models.CharField(
        max_length=20, choices=PERSONALITY_CHOICES, blank=True, null=True
    )
    cleanliness = models.CharField(
        max_length=10, choices=CLEANLINESS_CHOICES, blank=True, null=True
    )
    smoking = models.CharField(
        max_length=15, choices=SMOKING_CHOICES, blank=True, null=True
    )
    budget_min = models.IntegerField(default=0)  # minimum monthly budget in EGP
    budget_max = models.IntegerField(default=0)  # maximum monthly budget in EGP
    room_type_preference = models.CharField(
        max_length=10, choices=ROOM_TYPE_CHOICES, blank=True, null=True
    )

    # ── Personal Preferences Section ─────────────────────────
    # What this student WANTS in their ideal roommate
    preferred_room_type = models.CharField(
        max_length=10, choices=ROOM_TYPE_CHOICES, blank=True, null=True
    )
    smoking_preference = models.CharField(
        max_length=15, choices=SMOKING_CHOICES, blank=True, null=True
    )
    sleep_schedule_pref = models.CharField(
        max_length=20, choices=SLEEPING_CHOICES, blank=True, null=True
    )
    cleanliness_pref = models.CharField(
        max_length=10, choices=CLEANLINESS_CHOICES, blank=True, null=True
    )
    personality_pref = models.CharField(
        max_length=20, choices=PERSONALITY_CHOICES, blank=True, null=True
    )

    def __str__(self):
        return f"{self.user.username} - Student Profile"


# ──────────────────────────────────────────────────────────────────────────────────────────


class LandlordProfile(models.Model):
    """
    Extra fields for landlords only.
    Auto-created by signal when a landlord registers.

    Used in: Owner Dashboard, Payments page
    """

    # One landlord = one profile
    user = models.OneToOneField(
        Users, on_delete=models.CASCADE, related_name="landlord_profile"
    )

    # ── Business Info ────────────────────────────────────────
    company_name = models.CharField(
        max_length=255, blank=True, null=True
    )  # ((optional)) business name
    national_id = models.CharField(max_length=50, blank=True, null=True)
    is_id_verified = models.BooleanField(
        default=False
    )  # admin verifies this manually **

    # ── Financial Info (used in Payments dashboard) ──────────
    # NOTE: These are calculated fields — consider computing from Payment model instead
    total_income = models.DecimalField(max_digits=12, decimal_places=2, default=0)
    available_balance = models.DecimalField(max_digits=12, decimal_places=2, default=0)

    def __str__(self):
        return f"{self.user.username} - Landlord Profile"


class VerificationDocument(models.Model):
    """
    Documents students upload to get verified.

    Shown on profile sidebar:
         National ID
         Student ID
         University Email

    (Admin) reviews and sets is_verified = True manually.
    """

    DOC_TYPES = [
        ("national_id", "National ID"),
        ("student_id", "Student ID"),
        ("university_email", "University Email"),
    ]

    user = models.ForeignKey(
        Users, on_delete=models.CASCADE, related_name="documents"
    )  # one user has many docs
    doc_type = models.CharField(max_length=20, choices=DOC_TYPES)
    file = models.FileField(upload_to="verifications/", null=True, blank=True)
    is_verified = models.BooleanField(default=False)  # (Admin) sets this after review
    uploaded_at = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return f"{self.user.username} - {self.doc_type}"

```

# ────────────────────────End Model──────────────────────────────────────



## messaging app overview ##


# ────────────────────────Start URL─────────────────────────────────────

```
from django.urls import path
from .views import (
    ConversationView,
    MessageView,
)

urlpatterns = [
    path("",                       ConversationView.as_view(),  name="conversations"),
    path("<int:conversation_id>/", MessageView.as_view(),   name="messages"),
]     
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```




from rest_framework import serializers

from messaging.models import Conversation, Message
from accounts.models import Users
from bookings.models import Booking
 




class MessageSerializer(serializers.ModelSerializer):
    sender_name = serializers.CharField(source="sender.get_full_name", read_only=True) #
    
    class Meta:
        model  = Message
        fields = ["id", "conversation", "sender", "sender_name", "body", "is_read", "created_at"]
        read_only_fields = fields

#────────────────────────────────────────────────────────────────────────────────────────────

class ConversationSerializer(serializers.ModelSerializer):
    last_message  = serializers.SerializerMethodField()# why i need the last message 
    unread_count  = serializers.SerializerMethodField()
    other_user    = serializers.SerializerMethodField()

    class Meta:
        model  = Conversation
        fields = ["id", "initiator", "receiver", "booking", "property",
                  "other_user", "last_message", "unread_count", "created_at", "updated_at"]
        read_only_fields = fields

    def get_last_message(self, obj):
        msg = obj.messages.last()
        if not msg:
            return None
        return {"body": msg.body, "created_at": msg.created_at, "sender": msg.sender_id}# sender_id from where

    def get_unread_count(self, obj):#we have in view??
        user = self.context["request"].user
        return obj.messages.filter(is_read=False).exclude(sender=user).count()

    def get_other_user(self, obj):
        user = self.context["request"].user
        other = obj.receiver if obj.initiator == user else obj.initiator
        return {"id": other.id, "name": other.get_full_name(), "profile_picture": str(other.profile_picture) if other.profile_picture else None}

#────────────────────────────────────────────────────────────────────────────────────────────

class StartConversationSerializer(serializers.Serializer):
    receiver_id = serializers.IntegerField()
    booking_id  = serializers.IntegerField(required=False)
    message     = serializers.CharField()

    def validate_receiver_id(self, value):
        try:
            Users.objects.get(id=value)
        except Users.DoesNotExist:
            raise serializers.ValidationError("User not found.")
        return value
```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```




from django.db.models import Q
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework.permissions import IsAuthenticated
from rest_framework import status

from messaging.models import Conversation, Message
from accounts.models import Users
from bookings.models import Booking
from api.messaging_api.serializers import (
    ConversationSerializer,
    MessageSerializer,
    StartConversationSerializer,
)



class ConversationView(APIView):
    """
    GET  /api/messages/  → list all my conversations
    POST /api/messages/  → start a new conversation (or return existing one)
 
    Both methods live on the same view so they share one URL cleanly.
    Django routes to get() or post() based on the HTTP method.
    """
    permission_classes = [IsAuthenticated]

    def get(self, request):
        conversations = Conversation.objects.filter(
            Q(initiator=request.user) | Q(receiver=request.user)
        ).prefetch_related("messages")
        serializer = ConversationSerializer(conversations, many=True, context={"request": request})
        return Response(serializer.data)
    
    def post(self, request):
        serializer = StartConversationSerializer(data=request.data)
        serializer.is_valid(raise_exception=True)

        receiver_id = serializer.validated_data.get("receiver_id")
        receiver    = Users.objects.get(id=receiver_id)
        booking_id  = serializer.validated_data.get("booking_id")
        booking     = Booking.objects.get(id=booking_id) if booking_id else None

         # Return existing conversation if one already exists for this pai
        conv = Conversation.objects.filter(
            Q(initiator=request.user, receiver=receiver) |
            Q(initiator=receiver,     receiver=request.user),
            booking=booking
        ).first()

        if not conv:
            conv = Conversation.objects.create(
                initiator=request.user,
                receiver=receiver,
                booking=booking,
                property=booking.property if booking else None,
            )

        # Create first message
        Message.objects.create(
            conversation=conv,
            sender=request.user,
            body=serializer.validated_data.get("message"),
        )

        return Response(
            ConversationSerializer(conv, context={"request": request}).data,
            status=status.HTTP_201_CREATED,
        )


#────────────────────────────────────────────────────────────────────────────────────────────

class MessageView(APIView):
    """
    GET /api/messages/<id>/ — load conversation history + mark messages read.
    """
    permission_classes = [IsAuthenticated]

    def get(self, request, conversation_id):
        try:
            conv = Conversation.objects.get(
                Q(initiator=request.user) | Q(receiver=request.user),
                id=conversation_id,
            )
        except Conversation.DoesNotExist:
            return Response({"error": "Conversation not found."}, status=status.HTTP_404_NOT_FOUND)

        # Mark all messages from the other person as read
        conv.messages.exclude(sender=request.user).update(is_read=True)

        messages   = conv.messages.select_related("sender")
        serializer = MessageSerializer(messages, many=True)
        return Response(serializer.data)


    
    
```

# ────────────────────────End View──────────────────────────────────────







# ────────────────────────Start Model────────────────────────────────────

```



from django.db import models

from accounts.models import Users
from bookings.models import Booking
from properties.models import Property



class Conversation(models.Model):
    """
    A conversation between two users.
    Can be standalone (DM) or linked to a booking/property.

    booking=None  → standalone DM
    booking=<id>  → linked to a specific booking (tenant ↔ landlord)
    """
    initiator = models.ForeignKey(Users, on_delete=models.CASCADE, related_name="initiated_conversations")
    receiver  = models.ForeignKey(Users, on_delete=models.CASCADE, related_name="received_conversations")

    booking  = models.ForeignKey(Booking,  on_delete=models.SET_NULL, null=True, blank=True, related_name="conversations")
    property = models.ForeignKey(Property, on_delete=models.SET_NULL, null=True, blank=True, related_name="conversations")

    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        ordering = ["-updated_at"]
        constraints = [
            # Prevent duplicate conversations for the same pair + booking
            models.UniqueConstraint(
                fields=["initiator", "receiver", "booking"],
                name="unique_conversation_per_booking",
            )
        ]

    def __str__(self):
        return f"{self.initiator.username} ↔ {self.receiver.username} ({self.booking or 'DM'})"
    
 
#─────────────────────────────────────────────────────────────────────────────────────────────────────────────

class Message(models.Model):
    conversation = models.ForeignKey(Conversation, on_delete=models.CASCADE, related_name="messages")
    sender       = models.ForeignKey(Users, on_delete=models.CASCADE, related_name="sent_messages")

    body    = models.TextField()
    is_read = models.BooleanField(default=False)

    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        ordering = ["created_at"]

    def __str__(self):
        return f"{self.sender.username}: {self.body[:40]}"

```

# ────────────────────────End Model──────────────────────────────────────



## messaging Websocket setup ##


# ────────────────────────Start Routing────────────────────────────────────
```
from django.urls import re_path
from messaging.consumers import ChatConsumer


# The client connects to: ws://localhost:8000/ws/chat/<conversation_id>/?token=<jwt>

websocket_urlpatterns = [
    re_path(r"^ws/chat/(?P<conversation_id>\d+)/$", ChatConsumer.as_asgi()),
]
```
# ────────────────────────End Routing──────────────────────────────────────

# ────────────────────────Start Middleware────────────────────────────────────
```
from urllib.parse import parse_qs


from channels.middleware import BaseMiddleware

from channels.db import database_sync_to_async
from django.contrib.auth.models import AnonymousUser
from rest_framework_simplejwt.tokens import AccessToken
from rest_framework_simplejwt.exceptions import InvalidToken , TokenError

from accounts.models import Users

@database_sync_to_async
def get_user_from_token(token_key):

    """
    Validates the JWT access token and returns the matching User.
    Returns AnonymousUser if the token is missing, invalid, or expired.
    """
    try:
        token = AccessToken(token_key)
        return Users.objects.get(id=token["user_id"])
    except (InvalidToken, TokenError, Users.DoesNotExist):
        return AnonymousUser()


class JWTAuthMiddleware(BaseMiddleware):

    """
    Reads the JWT token from the WebSocket query string.

    The browser connects like:
        ws://localhost:8000/ws/chat/5/?token=eyJ...

    Normal DRF authentication (Authorization header) does not work for
    WebSocket connections — headers are not accessible the same way.
    Passing the token as a query param is the standard workaround.
    """
    async def __call__(self, scope, receive, send):
        query_string = scope.get("query_string", b"").decode()
        params       = parse_qs(query_string)
        token_list   = params.get("token",[])
        token        = token_list[0] if token_list else None

        scope["user"] = (
            await get_user_from_token(token)
            if token
            else AnonymousUser()
        )

        return await super().__call__(scope, receive, send)
```
# ────────────────────────End Middleware──────────────────────────────────────





# ────────────────────────Start Consumers────────────────────────────────────
```
import json

from channels.generic.websocket import AsyncWebsocketConsumer
from channels.db import database_sync_to_async
from django.contrib.auth.models import AnonymousUser
from django.db.models import Q

class ChatConsumer(AsyncWebsocketConsumer):
    """
    Handles one WebSocket connection for a single conversation room.

    Channel group name: "chat_<conversation_id>"
    All members connected to the same conversation share this group.
    When any member sends a message, every member in the group receives it.

    Flow:
        connect()    → verify JWT user → verify membership → join group → accept
        receive()    → save message to DB → broadcast to group
        chat_message() → push event to this specific WebSocket client
        disconnect() → leave group
    """    
    async def connect(self):
        self.conversation_id = self.scope["url_route"]["kwargs"]["conversation_id"]
        self.room_group_name = f"chat_{self.conversation_id}"
        self.user            = self.scope["user"]

        # Reject anonymous connections immediately
        if isinstance(self.user, AnonymousUser):
            await self.close()
            return
        # Reject users who are not part of this conversation
        if not await self._user_in_conversation():
            await self.close()
            return
        
        await self.channel_layer.group_add(self.room_group_name,self.channel_name)
        await self.accept()


    # ─────── receive ───────────────────────────────────────────────────────────
          
    async def receive(self, text_data):
        """
        Called when the connected client sends a message over the WebSocket.
        Saves it to the database then broadcasts it to the entire group.
        """

        try:
            data=json.loads(text_data)
        except json.JSONDecodeError:
            return
        
        body= data.get("body","").strip()
        if not body:
            return
        
        message = await self._save_message(body)

        await self.channel_layer.group_send(
            self.room_group_name,
            {
                "type":       "chat_message",
                "id":          message.id,
                "body":        message.body,
                "sender_id":   self.user.id, 
                "sender_name": await self._get_full_name(),
                "created_at" : str(message.created_at), 
            },
        )

    # ─────── send ───────────────────────────────────────────────────────────  
    async def chat_message(self,event):
        """
        Called by the channel layer when group_send delivers an event
        with type="chat_message". Pushes the payload to the WebSocket client.
        """ 
        await self.send(text_data=json.dumps(event))


    # ── Private DB helpers ────────────────────────────────────────────────────

    @database_sync_to_async
    def _user_in_conversation(self):
        from .models import Conversation
        return Conversation.objects.filter(
            Q(initiator=self.user) | Q(receiver=self.user),
            id=self.conversation_id,
        ).exists()
    
    @database_sync_to_async
    def _save_message(self,body):
        from .models import Conversation ,Message
        conversation=Conversation.objects.get(id=self.conversation_id)
        return Message.objects.create( 
            conversation=conversation,
            sender=self.user,
            body=body,
        )

    @database_sync_to_async
    def _get_full_name(self):
        return self.user.get_full_name()

    # ─────── disconnect ───────────────────────────────────────────────────────────    
    async def disconnect(self,close_code):
        await self.channel_layer.group_discard(self.room_group_name,self.channel_name)
```
# ────────────────────────End Consumers──────────────────────────────────────





# ────────────────────────Start Asgi────────────────────────────────────
```
"""
ASGI entry point for StudentHub.

Replaces the default wsgi.py as the server gateway when running with
Daphne or Uvicorn. Routes incoming connections by protocol:

    HTTP      → Django URL router → DRF views  (same as before)
    WebSocket → JWTAuthMiddleware → ChatConsumer

wsgi.py can stay in the project — it is still used by some deployment
tools for the HTTP-only path.
"""

import os
from django.core.asgi import get_asgi_application

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

# get_asgi_application() must be called before any models or consumers
# are imported so Django's app registry is fully loaded first.
django_asgi_app = get_asgi_application()

from channels.routing import ProtocolTypeRouter, URLRouter
from channels.security.websocket import AllowedHostsOriginValidator
from messaging.middleware import JWTAuthMiddleware
from messaging.routing      import websocket_urlpatterns as chat_patterns
from notifications.routing  import websocket_urlpatterns as notification_patterns

application = ProtocolTypeRouter({
    # All normal HTTP traffic goes through Django as usual
    "http": django_asgi_app,

    # WebSocket connections are authenticated first, then routed to consumers
    "websocket":# AllowedHostsOriginValidator(
        JWTAuthMiddleware(
            URLRouter(
                chat_patterns + notification_patterns   
            )
        )
    #),
})

```
# ────────────────────────End Asgi──────────────────────────────────────









# ────────────────────────Start Apps────────────────────────────────────
```
from django.apps import AppConfig


class MessagingConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name = "messaging"

```
# ────────────────────────End Apps──────────────────────────────────────















## booking overview


# ────────────────────────Start URL─────────────────────────────────────

```
"""Bookings API URL configuration."""
from django.urls import path
from api.bookings_api.views import (
    BookingCreateView,
    MyBookingView,
    BookingStatusView,
)

urlpatterns = [

    path(""                        , BookingCreateView.as_view() , name="booking-create"),
    path("my/"                     , MyBookingView.as_view() ,     name="booking-my"),
    path("<int:pk>/status/"        , BookingStatusView.as_view() , name="booking-status"),
]
        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```
"""
    Serializers
        -BookingSerializer
        -BookingCreateSerializer
        -BookingStatusSerializer
"""

from rest_framework import serializers 
from bookings.models import Booking




class BookingSerializer(serializers.ModelSerializer):

    class Meta:
        model=Booking
        fields = [  
                    "id" ,
                    "property" ,
                    "tenant" ,
                    "status" ,
                    "move_in_date" ,
                    "duration_months" ,
                    "message" ,
                    "total_amount_cents",        
                    "deposit_amount_cents",      
                    "remaining_amount_cents",    
                    "expires_at", 
                    "created_at" ,
                    "updated_at",
                    ]
        read_only_fields =[
            "id",
            "tenant",
            "status",
            "total_amount_cents",        
            "deposit_amount_cents",      
            "remaining_amount_cents",    
            "expires_at", 
            "created_at",
            "updated_at",
        ]

class BookingCreateSerializer(serializers.ModelSerializer):

    class Meta:
        model=Booking
        fields= [ "property" , "move_in_date" , "duration_months" , "message" ]
        
    def validate(self , data):
        prop            = data.get("property")
        duration_months = data.get("duration_months")

        if prop.min_stay_months and duration_months < prop.min_stay_months:
            raise serializers.ValidationError(f"This property requires a minimum stay of {prop.min_stay_months} months.")
        
        if prop.max_stay_months and duration_months > prop.max_stay_months:
            raise serializers.ValidationError(f"This property requires a maximum stay of {prop.max_stay_months} months.")
        

        return data


class BookingStatusSerializer(serializers.ModelSerializer):

    class Meta:
        model=Booking
        fields= [ "status" ]
```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```
"""
View 
    -BookingList
    -BookingCreateView
    -BookingStatusUpdate
"""

from django.db import transaction
from django.utils import timezone
from datetime import timedelta

from rest_framework.views import APIView
from rest_framework.response import Response 
from rest_framework import status 
from rest_framework.permissions import  IsAuthenticated
from api.accounts_api.permissions import IsStudent

from bookings.models import Booking
from properties.models import Property

from api.bookings_api.serializers import (
    BookingSerializer,
    BookingCreateSerializer,
    BookingStatusSerializer,
)



class MyBookingView(APIView):
    permission_classes = [IsAuthenticated]

    def get( self , request ):

        if request.user.role == "student":
            booking = Booking.objects.filter( tenant= request.user)
        else:
            booking = Booking.objects.filter( property__landlord= request.user)

        serilaizer = BookingSerializer( booking , many=True )
            
        return Response( serilaizer.data )



class BookingCreateView(APIView):
    """
    Booking creation flow:

    1. Validate incoming request data using BookingCreateSerializer.
    2. Start a database transaction to ensure data consistency.
    3. Lock the selected property row using select_for_update()
    to prevent concurrent reservations for the same property.
    4. Check if the property is still available.
    5. Check if the student already has an active booking
    for this property.
    6. Calculate booking financial snapshot:
        - total amount
        - deposit amount (20%)
        - remaining amount
    7. Create booking with temporary expiration time
    and status "pending_payment".
    8. Mark property as "reserved".
    9. Serialize created booking and return response data.
    """
    permission_classes = [IsStudent]
    
    def post( self , request ):
        serializer = BookingCreateSerializer( data = request.data , context={"request":request} )
        serializer.is_valid(raise_exception=True)

        property_id=serializer.validated_data["property"].id
        
        try:
            #transaction.atomic()= "treat these operations as one safe package"
            with transaction.atomic():
                #select_for_update()= "lock this row so others must wait"
                prop = Property.objects.select_for_update().get(id=property_id)

                if prop.status != "available":
                    return Response( {"error":"This property is no longer available."} , 
                                      status=status.HTTP_409_CONFLICT)
                
                already_exists=Booking.objects.filter(tenant=request.user , property=prop , status__in=["pending_payment", "deposit_paid", "confirmed"]).exists()
                if already_exists:
                    return Response(
                        {"error": "You already have an active booking for this property."},
                        status=status.HTTP_409_CONFLICT
                    )
                
                # Calculate financial snapshot from property price
                total     = int(prop.price * 100)
                deposit   = int(total * 0.20)
                remaining = total - deposit

                booking = serializer.save(
                            tenant                 = request.user,
                            total_amount_cents     = total,
                            deposit_amount_cents   = deposit,
                            remaining_amount_cents = remaining,
                            expires_at             = timezone.now() + timedelta(minutes=30),
                            status                 = "pending_payment",
                    )
                
                prop.status = "reserved"
                prop.save()

        except Property.DoesNotExist:
            return Response(
                {"error": "Property not found."},
                status=status.HTTP_404_NOT_FOUND
            )
        return Response( BookingSerializer( booking ).data , status=status.HTTP_201_CREATED )



class BookingStatusView(APIView):
    """
    Booking status update flow:

    1. Retrieve booking by ID.
    2. Verify booking exists.
    3. Determine user role (landlord or tenant).
    4. Validate that the user has permission
    to modify this booking.
    5. Define allowed status transitions
    based on the current booking status
    and user role.
    6. Validate requested new status.
    7. Start database transaction to ensure
    consistent booking/property updates.
    8. Update booking status using serializer validation.
    9. Synchronize property status:
        - cancelled  -> property becomes available
        - confirmed  -> property becomes rented
    10. Serialize updated booking and return response data.
    """
    permission_classes = [IsAuthenticated]


    def patch( self , request , pk ):

        try:
            booking = Booking.objects.get( pk=pk )
        except Booking.DoesNotExist:
            return Response({"error": "Booking not found."}, status=status.HTTP_404_NOT_FOUND)
        
        
        if request.user.role == "landlord":
            if request.user != booking.property.landlord:
                return Response({"error": "You do not own this property."}, status=status.HTTP_403_FORBIDDEN)
            
            allowed_transitions = {
                "deposit_paid": ["confirmed", "cancelled"],   # landlord confirms after deposit
                "confirmed":    ["completed"],                # landlord marks stay as done
            }

        else:
            if booking.tenant != request.user:
                return Response({"error": "This is not your booking."}, status=status.HTTP_403_FORBIDDEN)
            
            allowed_transitions = {
                "pending_payment": ["cancelled"],             # student cancels before paying
                "deposit_paid":    ["cancelled"],             # student cancels after deposit (refund logic goes here later)
            }


        current = booking.status
        allowed = allowed_transitions.get(current, [])
        new_status = request.data.get("status")
        if new_status not in allowed:
            return Response(
                {
                    "error": f"Cannot change status from '{current}' to '{new_status}'.",
                    "allowed": allowed,
                },
                status=status.HTTP_400_BAD_REQUEST
            )

        with transaction.atomic():
            serializer = BookingStatusSerializer(
                booking,
                data={"status": new_status},
                partial=True
            )
            serializer.is_valid(raise_exception=True)
            booking = serializer.save()

            # Free up the property if booking is cancelled
            if new_status == "cancelled":
                booking.property.status = "available"
                booking.property.save()

            # Mark property as fully rented when confirmed
            if new_status == "confirmed":
                booking.property.status = "rented"
                booking.property.save()

        return Response(
            BookingSerializer(booking).data,
            status=status.HTTP_200_OK
        )

```

# ────────────────────────End View──────────────────────────────────────





# ────────────────────────Start Signals────────────────────────────────────
```
"""
Bookings app signals.
Keeps property status in sync when a booking changes status.
"""

from django.db.models.signals import post_save
from django.dispatch import receiver
from bookings.models import Booking


@receiver(post_save, sender=Booking)
def sync_property_status(sender, instance, **kwargs):
    """
    Automatically updates the property's status based on booking outcome.

    - approved  → property becomes 'rented'   (no longer searchable)
    - cancelled → property reverts to 'available' (back in listings)
    - rejected  → property reverts to 'available'
    """
    prop = instance.property

    if instance.status == "approved":
        if prop.status != "rented":
            prop.status = "rented"
            prop.save(update_fields=["status"])

    elif instance.status in ("cancelled", "rejected"):
        if prop.status == "rented":
            prop.status = "available"
            prop.save(update_fields=["status"])

```
# ────────────────────────End Signals──────────────────────────────────────





# ────────────────────────Start Apps────────────────────────────────────
```
"""Without ready() importing signals, signals.py exists but never fires."""
from django.apps import AppConfig


class BookingsConfig(AppConfig):
    """Configuration class for the bookings app."""
    default_auto_field = "django.db.models.BigAutoField"
    name = "bookings"

    def ready(self):
        import bookings.signals  # noqa: F401 — registers signal handlers

```
# ────────────────────────End Apps──────────────────────────────────────






# ────────────────────────Start Model────────────────────────────────────

```
"""
booking models 
    -handel all room booking BTW student and landloard

"""
from django.utils import timezone
from datetime import timedelta
import builtins 

from django.db import models
from accounts.models import Users
from properties.models import Property


class Booking(models.Model):


    STATUS_CHOICES = [
        ("pending_payment", "Pending Payment"),
        ("deposit_paid", "Deposit Paid"),
        ("confirmed", "Confirmed"),
        ("cancelled", "Cancelled"),
        ("completed", "Completed"),
        ("expired", "Expired"),
    ]

    # ────Parties──────────────────────────────────────────────────────────────────────────────────────────────────────────
    tenant     = models.ForeignKey(Users, on_delete=models.CASCADE, related_name="bookings" , limit_choices_to={ "role" : "student" })
    property   = models.ForeignKey(Property, on_delete=models.CASCADE, related_name="bookings")

    # ────Booking Details─────────────────────────────────────────────────────────────────────────────────────────────────
    status          = models.CharField(max_length=15 ,choices=STATUS_CHOICES ,default='pending_payment')
    move_in_date    = models.DateField()
    duration_months = models.PositiveIntegerField()
    message         = models.TextField(null=True , blank=True)

    # ── Financial Snapshot ───────────────────────────────────
    # Copied from property.price at booking time — never changes even if landlord later edits the listing price
    total_amount_cents   = models.PositiveIntegerField()    # full rent agreed
    deposit_amount_cents = models.PositiveIntegerField()    # what student pays now
    remaining_amount_cents = models.PositiveIntegerField()  # what's left after deposit

    # ── Expiration ───────────────────────────────────────────
    # If student doesn't pay deposit within 30 min, booking auto-expires
    expires_at = models.DateTimeField()

    # ── Timestamps ───────────────────────────────────────────
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    #TODO: custom validation to prevent the tenant booking the same property more than one time

    class Meta:
        ordering = ["-created_at"]

    def __str__(self):
        return f"{self.tenant.username} - {self.property.title} -({self.status})"
    
    def save(self, *args, **kwargs):
        # Auto-set expiry to 30 min from creation on first save
        if not self.pk and not self.expires_at:
            self.expires_at = timezone.now() + timedelta(minutes=30)
        super().save(*args, **kwargs)
    
    @builtins.property
    def is_expired(self):
        return self.status == "pending_payment" and timezone.now() > self.expires_at
```

# ────────────────────────End Model──────────────────────────────────────


## properties app ##


# ────────────────────────Start URL─────────────────────────────────────

```
"""
    Properties API URL configuration

"""
from django.urls import path
from api.properties_api.views import (
    PropertyListView,
    FeaturedPropertiesView,
    UniversityPropertiesView,
    PropertyDetailView,
    PropertyCreateView,
    PropertyEditView,
    PropertyImageUploadView,
    PropertyImageDeleteView,
    LandlordPropertiesView,
)

urlpatterns = [
    # ── Public Listings ───────────────────────────────────────────────────────
    path(""                   , PropertyListView.as_view(),         name="property-list"),
    path("featured/"          , FeaturedPropertiesView.as_view(),   name="property-featured"),
    path("university/"        , UniversityPropertiesView.as_view(), name="property-university"),
    path("<int:property_id>/" , PropertyDetailView.as_view(),       name="property-detail"),

    # ── Landlord — Listing Management ─────────────────────────────────────────
    path("create/"                 , PropertyCreateView.as_view(),   name="property-create"),
    path("<int:property_id>/edit/" , PropertyEditView.as_view(),     name="property-edit"),

    # ── Landlord — Image Management ───────────────────────────────────────────
    path("<int:property_id>/images/"                , PropertyImageUploadView.as_view(), name="property-image-upload"),
    path("<int:property_id>/images/<int:image_id>/" , PropertyImageDeleteView.as_view(), name="property-image-delete"),

    # ── Owner Dashboard ───────────────────────────────────────────────────────
    path("landlord/properties/", LandlordPropertiesView.as_view(), name="landlord-properties"),
]

        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```
"""
    Properties API serializers.

    Serializers:
        - PropertyImageSerializer   → nested images list in PropertySerializer
        - PropertySerializer        → full detail for GET /api/properties/<id>/
        - PropertyListSerializer    → lightweight card view for GET /api/properties/
        - PropertyCreateSerializer  → POST /api/properties/create/
        - PropertyUpdateSerializer  → PATCH /api/properties/<id>/edit/

"""
from rest_framework import serializers
from properties.models import Property, PropertyImage


class PropertyImageSerializer(serializers.ModelSerializer):
    """Nested inside PropertySerializer. Cover image floats first (model-level ordering)."""

    class Meta:
        model            = PropertyImage
        fields           = ["id", "image", "is_cover", "uploaded_at"]
        read_only_fields = ["id", "uploaded_at"]


class PropertySerializer(serializers.ModelSerializer):
    """
    Full detail view — used on the property detail page.
    Includes nested images, owner info, and computed rating fields.
    """

    # used many=True because one property has many images (reverse relation)
    images = PropertyImageSerializer(many=True, read_only=True)

    # Computed fields from model properties
    average_rating = serializers.FloatField(read_only=True)
    review_count   = serializers.IntegerField(read_only=True)

    # Landlord info — needed when showing property (ForeignKey relation).
    # Only expose specific fields to avoid exposing sensitive User data.
    landlord_id           = serializers.IntegerField(source="landlord.id", read_only=True)
    landlord_name         = serializers.CharField(source="landlord.get_full_name", read_only=True)
    landlord_picture      = serializers.ImageField(source="landlord.profile_picture", read_only=True)
    landlord_is_verified  = serializers.BooleanField(source="landlord.is_verified", read_only=True)
    landlord_is_top_rated = serializers.BooleanField(source="landlord.is_top_rated", read_only=True)
   
    # Computed at request time by PropertyDetailView — not stored in DB
    university_distance = serializers.SerializerMethodField()

    class Meta:
        model = Property
        fields = [
            "id",
            # ── Ownership ────────────────────────────────────
            "landlord_id", "landlord_name", "landlord_picture",
            "landlord_is_verified", "landlord_is_top_rated",
            # ── Basic Info ───────────────────────────────────
            "title", "description", "property_type",
            # ── Pricing ──────────────────────────────────────
            "price",
            # ── Location ─────────────────────────────────────
            "city", "district", "address", "latitude", "longitude",
            # ── University Proximity ─────────────────────────
            "nearby_university", "distance_to_university", "transport_type","university_distance",
            # ── Room Details ─────────────────────────────────
            "num_rooms", "num_beds", "num_bathrooms", "num_roommates",
            "floor", "area_sqm", "gender_preference",
            # ── Amenities ────────────────────────────────────
            "amenities",
            # ── Stay Duration ────────────────────────────────
            "min_stay_months", "max_stay_months",
            # ── Status & Visibility ──────────────────────────
            "status", "is_featured",
            # ── Analytics ────────────────────────────────────
            "view_count",
            # ── Reviews (computed) ───────────────────────────
            "average_rating", "review_count",
            # ── Images ───────────────────────────────────────
            "images",
            # ── Timestamps ───────────────────────────────────
            "created_at", "updated_at",
        ]
        read_only_fields = ["id", "view_count", "is_featured", "created_at", "updated_at"]

    def get_university_distance(self, obj):
        """
        Returns Google Distance Matrix result if injected by the view.
        Falls back to None gracefully so serializer never crashes.

        The view injects it via:
            PropertySerializer(prop, context={"request": req, "university_distance": result})
        """
        return self.context.get("university_distance", None)


class PropertyListSerializer(serializers.ModelSerializer):
    """
    Lightweight card view — used on FindRoom and Home page listing grids.
    Only the fields needed to render a card. Keeps list responses fast.
    """

    cover_image          = serializers.SerializerMethodField()
    average_rating       = serializers.FloatField(read_only=True)
    review_count         = serializers.IntegerField(read_only=True)
    landlord_name        = serializers.CharField(source="landlord.get_full_name", read_only=True)
    landlord_is_verified = serializers.BooleanField(source="landlord.is_verified", read_only=True)

    class Meta:
        model  = Property
        fields = [
            "id", "title", "property_type", "price",
            "city", "district",
            "nearby_university", "distance_to_university", "transport_type",
            "num_beds", "num_roommates", "gender_preference",
            "amenities", "status", "is_featured",
            "average_rating", "review_count",
            "cover_image",
            "landlord_name", "landlord_is_verified",
            "created_at",
        ]

    def get_cover_image(self, obj):
        """Returns the URL of the cover image, or None if no images uploaded."""
        cover = obj.images.filter(is_cover=True).first() or obj.images.first()
        if not cover:
            return None
        request = self.context.get("request")
        # Build absolute URL so frontend doesn't need to prefix the media root
        # request.build_absolute_uri() adds full domain (e.g., http://localhost:8000) to URL
        # Without it, frontend only gets "/media/image.jpg" which may not work
        return request.build_absolute_uri(cover.image.url) if request else cover.image.url



class PropertyCreateSerializer(serializers.ModelSerializer):
    """
    POST /api/properties/create/ — landlord creates a new listing.
    owner is injected in the view via save(owner=request.user), not sent by client.
    
    """

    # Accept uploaded images at creation time (optional)
    uploaded_images = serializers.ListField(child=serializers.ImageField(),write_only=True,required=False,)

    class Meta:
        model = Property
        fields = [
            "title", "description", "property_type",
            "price",
            "city", "district", "address", "latitude", "longitude",
            "nearby_university", "distance_to_university", "transport_type",
            "num_rooms", "num_beds", "num_bathrooms", "num_roommates",
            "floor", "area_sqm", "gender_preference",
            "amenities",
            "min_stay_months", "max_stay_months",
            "status",
            "uploaded_images",
        ]

    def validate_price(self, value):
        if value <= 0:
            raise serializers.ValidationError("Price must be greater than 0.")
        return value

    def validate(self, data):
        min_stay = data.get("min_stay_months", 1)
        max_stay = data.get("max_stay_months")
        if max_stay and max_stay < min_stay:
            raise serializers.ValidationError("max_stay_months cannot be less than min_stay_months.")
        return data

    def create(self, validated_data):
        images = validated_data.pop("uploaded_images", [])
        property_obj = Property.objects.create(**validated_data)

        for index, image_file in enumerate(images):
            PropertyImage.objects.create(
                property=property_obj,
                image=image_file,
                is_cover=(index == 0),  # first uploaded image is the cover
            )

        return property_obj


class PropertyUpdateSerializer(serializers.ModelSerializer):
    """
    PATCH /api/properties/<id>/edit/ — landlord edits their own listing.
    All fields optional. Images managed separately via PropertyImageSerializer.
    """

    class Meta:
        model = Property
        fields = [
            "title", "description", "property_type",
            "price",
            "city", "district", "address", "latitude", "longitude",
            "nearby_university", "distance_to_university", "transport_type",
            "num_rooms", "num_beds", "num_bathrooms", "num_roommates",
            "floor", "area_sqm", "gender_preference",
            "amenities",
            "min_stay_months", "max_stay_months",
            "status",
        ]

    def validate_price(self, value):
        if value <= 0:
            raise serializers.ValidationError("Price must be greater than 0.")
        return value

    def validate(self, data):
        # On partial update, pull existing values if not being changed
        instance = self.instance
        min_stay = data.get("min_stay_months", instance.min_stay_months)
        max_stay = data.get("max_stay_months", instance.max_stay_months)
        if max_stay and max_stay < min_stay:
            raise serializers.ValidationError("max_stay_months cannot be less than min_stay_months.")
        return data


```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```
"""
Properties API views.

Views:
    - PropertyListView          → GET  /api/properties/            (filtered list)
    - FeaturedPropertiesView    → GET  /api/properties/featured/   (is_featured=True)
    - UniversityPropertiesView  → GET  /api/properties/university/ (filter by nearby_university)
    - PropertyDetailView        → GET  /api/properties/<id>/       (increments view_count)
    - PropertyCreateView        → POST /api/properties/create/     (landlords only)
    - PropertyEditView          → PATCH /api/properties/<id>/edit/ (landlord only)
"""
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework.permissions import IsAuthenticated, IsAuthenticatedOrReadOnly
from rest_framework import status
from django.db.models import Q
from properties.models import Property, PropertyImage
from api.accounts_api.permissions import IsLandlord
from api.properties_api.serializers import (
    PropertySerializer,
    PropertyListSerializer,
    PropertyCreateSerializer,
    PropertyUpdateSerializer,
    PropertyImageSerializer,
)

import logging

from services.google_maps import (
    get_distance_to_university,
    GoogleMapsError,
)

logger = logging.getLogger(__name__)

TRANSPORT_TO_MODE = {
    "walk": "walking",
    "metro": "transit",
    "transport": "transit",
}

# ── Helpers ───────────────────────────────────────────────────────────────────

def apply_filters(queryset, params):
    """
    Applies query-param filters to a Property queryset.
    Called by PropertyListView and UniversityPropertiesView.

    Supported params:
        city, district, type (property_type), status,
        price_min, price_max, num_beds, num_rooms,
        gender, university, amenity (single value, checks if list contains it)
    """
    city = params.get("city")
    district = params.get("district")
    property_type = params.get("type")
    prop_status = params.get("status")
    price_min = params.get("price_min")
    price_max = params.get("price_max")
    num_beds = params.get("num_beds")
    num_rooms = params.get("num_rooms")
    gender = params.get("gender")
    university = params.get("university")
    amenity = params.get("amenity")

    if city:
        queryset = queryset.filter(city__icontains=city)
    if district:
        queryset = queryset.filter(district__icontains=district)
    if property_type:
        queryset = queryset.filter(property_type=property_type)
    if prop_status:
        queryset = queryset.filter(status=prop_status)
    if price_min:
        queryset = queryset.filter(price__gte=price_min)
    if price_max:
        queryset = queryset.filter(price__lte=price_max)
    if num_beds:
        queryset = queryset.filter(num_beds=num_beds)
    if num_rooms:
        queryset = queryset.filter(num_rooms=num_rooms)
    if gender:
        queryset = queryset.filter(gender_preference=gender)
    if university:
        queryset = queryset.filter(nearby_university__icontains=university)
    if amenity:
        # JSONField list — filter rows where the amenity string appears in the array
        queryset = queryset.filter(amenities__contains=amenity)

    return queryset


# ── Views ─────────────────────────────────────────────────────────────────────

class PropertyListView(APIView):
    """
    GET /api/properties/
    Public listing. Returns all available properties with optional filters.

    Example: /api/properties/?city=Cairo&type=studio&price_min=1000&num_beds=2
    """
    permission_classes = [IsAuthenticatedOrReadOnly]

    def get(self, request):
        queryset = Property.objects.select_related("landlord").prefetch_related("images")

        # Default to available only — frontend can pass status=rented to override
        if not request.query_params.get("status"):
            queryset = queryset.filter(status="available")

        queryset = apply_filters(queryset, request.query_params)
        serializer = PropertyListSerializer(queryset, many=True, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)


class FeaturedPropertiesView(APIView):
    """
    GET /api/properties/featured/
    Returns is_featured=True available listings.
    Used in the 'Featured Properties' section on the home page.
    """
    permission_classes = [IsAuthenticatedOrReadOnly]

    def get(self, request):
        queryset = (
            Property.objects
            .filter(is_featured=True, status="available")
            .select_related("landlord")
            .prefetch_related("images")
        )
        serializer = PropertyListSerializer(queryset, many=True, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)


class UniversityPropertiesView(APIView):
    """
    GET /api/properties/university/?university=Cairo+University
    Returns listings near a specific university.
    Falls back to all properties with a university set if no param provided.
    """
    permission_classes = [IsAuthenticatedOrReadOnly]

    def get(self, request):
        queryset = (
            Property.objects
            .filter(status="available")
            .exclude(nearby_university__isnull=True)
            .exclude(nearby_university="")
            .select_related("landlord")
            .prefetch_related("images")
        )
        queryset = apply_filters(queryset, request.query_params)
        serializer = PropertyListSerializer(queryset, many=True, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)


class PropertyDetailView(APIView):
    """
    GET /api/properties/<id>/
    Full detail page. Increments view_count on every request.
    Public — no auth required to browse.
    """
    permission_classes = [IsAuthenticatedOrReadOnly]

    def get(self, request, property_id):
        try:
            prop = (
                Property.objects
                .select_related("landlord")
                .prefetch_related("images")
                .get(id=property_id)
            )
        except Property.DoesNotExist:
            return Response({"error": "Property not found."}, status=status.HTTP_404_NOT_FOUND)

        # NOTE: use F() to avoid race conditions if we ever add concurrency
        from django.db.models import F
        Property.objects.filter(id=property_id).update(view_count=F("view_count") + 1)
        prop.refresh_from_db(fields=["view_count"])

        university_distance = None

        if prop.latitude and prop.longitude and prop.nearby_university:
            mode = TRANSPORT_TO_MODE.get(
                prop.transport_type,
                "walking"
            )

            try:
                university_distance = get_distance_to_university(
                    origin_lat=float(prop.latitude),
                    origin_lng=float(prop.longitude),
                    destination=prop.nearby_university,
                    mode=mode,
                )
            except GoogleMapsError as exc:
                logger.warning(
                    "Distance Matrix failed for property %s: %s",
                    property_id,
                    exc,
                )

        serializer = PropertySerializer(
            prop,
            context={
                "request": request,
                "university_distance": university_distance,
            },
        )
        return Response(serializer.data, status=status.HTTP_200_OK)


class PropertyCreateView(APIView):
    """
    POST /api/properties/create/
    Landlords only. landlord is injected server-side — never trust the client to send it.
    """
    permission_classes = [IsLandlord]

    def post(self, request):
        serializer = PropertyCreateSerializer(data=request.data)
        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        prop = serializer.save(landlord=request.user)

        # Return full detail so the frontend can immediately show the new listing
        return Response(
            PropertySerializer(prop, context={"request": request}).data,
            status=status.HTTP_201_CREATED,
        )


class PropertyEditView(APIView):
    """
    PATCH /api/properties/<id>/edit/
    Owner only. Landlord can only edit their own listings.
    """
    permission_classes = [IsLandlord]

    def patch(self, request, property_id):
        try:
            prop = Property.objects.get(id=property_id)
        except Property.DoesNotExist:
            return Response({"error": "Property not found."}, status=status.HTTP_404_NOT_FOUND)

        # Prevent landlords from editing other landlords' listings
        if prop.landlord != request.user:
            return Response({"error": "You do not own this property."}, status=status.HTTP_403_FORBIDDEN)

        serializer = PropertyUpdateSerializer(prop, data=request.data, partial=True)
        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        serializer.save()
        return Response(
            PropertySerializer(prop, context={"request": request}).data,
            status=status.HTTP_200_OK,
        )


class PropertyImageUploadView(APIView):
    """
    POST /api/properties/<id>/images/
    Add images to an existing listing. Only the landlord can upload.
    Send images as multipart/form-data with key 'images' (multiple files allowed).
    """
    permission_classes = [IsLandlord]

    def post(self, request, property_id):
        try:
            prop = Property.objects.get(id=property_id)
        except Property.DoesNotExist:
            return Response({"error": "Property not found."}, status=status.HTTP_404_NOT_FOUND)

        if prop.landlord != request.user:
            return Response({"error": "You do not own this property."}, status=status.HTTP_403_FORBIDDEN)

        images = request.FILES.getlist("images")
        if not images:
            return Response({"error": "No images provided."}, status=status.HTTP_400_BAD_REQUEST)

        has_cover = prop.images.filter(is_cover=True).exists()
        created = []

        for index, image_file in enumerate(images):
            # First image becomes cover only if property has no cover yet
            is_cover = (index == 0) and not has_cover
            img = PropertyImage.objects.create(property=prop, image=image_file, is_cover=is_cover)
            created.append(img)

        return Response(
            PropertyImageSerializer(created, many=True, context={"request": request}).data,
            status=status.HTTP_201_CREATED,
        )


class PropertyImageDeleteView(APIView):
    """
    DELETE /api/properties/<id>/images/<image_id>/
    Owner removes a single image. If the deleted image was the cover,
    the next oldest image is promoted automatically.
    """
    permission_classes = [IsLandlord]

    def delete(self, request, property_id, image_id):
        try:
            prop = Property.objects.get(id=property_id)
        except Property.DoesNotExist:
            return Response({"error": "Property not found."}, status=status.HTTP_404_NOT_FOUND)

        if prop.landlord != request.user:
            return Response({"error": "You do not own this property."}, status=status.HTTP_403_FORBIDDEN)

        try:
            image = PropertyImage.objects.get(id=image_id, property=prop)
        except PropertyImage.DoesNotExist:
            return Response({"error": "Image not found."}, status=status.HTTP_404_NOT_FOUND)

        was_cover = image.is_cover
        image.delete()

        # Promote oldest remaining image as cover if we just deleted the cover
        if was_cover:
            next_image = prop.images.order_by("uploaded_at").first()
            if next_image:
                next_image.is_cover = True
                next_image.save()

        return Response(status=status.HTTP_204_NO_CONTENT)


class LandlordPropertiesView(APIView):
    """
    GET /api/owner/properties/
    Returns all properties belonging to the logged-in landlord.
    Used in the landlord dashboard 'My Properties' tab.
    """
    permission_classes = [IsLandlord]

    def get(self, request):
        queryset = (
            Property.objects
            .filter(landlord=request.user)
            .prefetch_related("images")
        )
        serializer = PropertyListSerializer(queryset, many=True, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)

```

# ────────────────────────End View──────────────────────────────────────


# ────────────────────────Start Signals────────────────────────────────────
```

"""
properties/signals.py

Auto-geocodes a property's address to lat/lng whenever:
  - A new property is created with an address
  - An existing property's address is changed

Uses Google Geocoding API via services.google_maps.geocode_address().
Fails silently — if geocoding fails the property is still saved,
just without coordinates. The landlord can retry by re-saving.
"""

import logging
from django.db.models.signals import pre_save
from django.dispatch import receiver
from properties.models import Property
from services.google_maps import geocode_address, GoogleMapsError

logger = logging.getLogger(__name__)


@receiver(pre_save, sender=Property)
def auto_geocode_address(sender, instance, **kwargs):
    """
    Fires before every Property save.

    Triggers geocoding when:
        1. New property (no pk yet) and has an address
        2. Existing property and address has changed

    Skips geocoding when:
        - No address provided
        - Address hasn't changed
        - lat/lng were manually provided by the landlord (we respect manual coords)
    """
    if not instance.address:
        return

    # Check if address changed (skip for new instances — they have no pk yet)
    address_changed = False
    if instance.pk:
        try:
            old = Property.objects.get(pk=instance.pk)
            address_changed = old.address != instance.address
        except Property.DoesNotExist:
            address_changed = True
    else:
        # New property
        address_changed = True

    if not address_changed:
        return

    # If landlord manually supplied lat/lng alongside a new address, respect them
    if instance.pk is None and instance.latitude and instance.longitude:
        return

    try:
        result = geocode_address(instance.address)
        instance.latitude  = result["lat"]
        instance.longitude = result["lng"]
        logger.info(
            "Geocoded property '%s': lat=%s, lng=%s",
            instance.title,
            result["lat"],
            result["lng"],
        )
    except GoogleMapsError as exc:
        # Log but don't block the save
        logger.warning(
            "Geocoding failed for property '%s' (address: %s): %s",
            instance.title,
            instance.address,
            exc,
        )
```
# ────────────────────────End Signals──────────────────────────────────────






# ────────────────────────Start Permissions────────────────────────────────────
```
"""
Properties API permissions.
Object-level permission so ownership checks don't live inside every view.
"""
from rest_framework.permissions import BasePermission, IsAuthenticated


class IsPropertyOwner(BasePermission):
    """
    Object-level permission — allows access only if the requesting user
    owns the property being accessed.

    Usage in views:
        permission_classes = [IsPropertyOwner]

        def patch(self, request, property_id):
            prop = get_object_or_404(Property, id=property_id)
            self.check_object_permissions(request, prop)   ← triggers has_object_permission
            ...
    """
    message = "You do not own this property."

    def has_permission(self, request, view):
        """User must be authenticated and be a landlord."""
        return bool(
            request.user
            and request.user.is_authenticated
            and request.user.role == "landlord"
        )

    def has_object_permission(self, request, view, obj):
        """obj is the Property instance. Checks landlord field directly."""
        return obj.landlord == request.user


```
# ────────────────────────End Permissions────────────────────────────────────




# ────────────────────────Start Filters────────────────────────────────────
```
"""
Properties API filters.
Replaces the manual apply_filters() helper in views.py.

Usage in views:
    from api.properties_api.filters import PropertyFilter

    queryset = Property.objects.all()
    filtered = PropertyFilter(request.query_params, queryset=queryset).qs

Supported query params:
    city, district, type, status,
    price_min, price_max,
    num_beds, num_rooms, num_bathrooms,
    gender, university,
    is_featured, amenity
"""
import django_filters
from properties.models import Property


class PropertyFilter(django_filters.FilterSet):
    """
    FilterSet for Property listings.
    Each filter maps a clean query-param name to the correct ORM lookup.
    """

    # ── Location ─────────────────────────────────────────────────────────────
    city = django_filters.CharFilter(field_name="city", lookup_expr="icontains")
    district = django_filters.CharFilter(field_name="district", lookup_expr="icontains")

    # ── Type & Status ─────────────────────────────────────────────────────────
    # 'type' is a reserved word in Python, so we alias it here
    type = django_filters.CharFilter(field_name="property_type", lookup_expr="exact")
    status = django_filters.CharFilter(field_name="status", lookup_expr="exact")

    # ── Pricing ───────────────────────────────────────────────────────────────
    price_min = django_filters.NumberFilter(field_name="price", lookup_expr="gte")
    price_max = django_filters.NumberFilter(field_name="price", lookup_expr="lte")

    # ── Room Details ──────────────────────────────────────────────────────────
    num_beds = django_filters.NumberFilter(field_name="num_beds", lookup_expr="exact")
    num_rooms = django_filters.NumberFilter(field_name="num_rooms", lookup_expr="exact")
    num_bathrooms = django_filters.NumberFilter(field_name="num_bathrooms", lookup_expr="exact")

    # ── Preferences ───────────────────────────────────────────────────────────
    gender = django_filters.CharFilter(field_name="gender_preference", lookup_expr="exact")

    # ── University ────────────────────────────────────────────────────────────
    university = django_filters.CharFilter(field_name="nearby_university", lookup_expr="icontains")

    # ── Visibility ────────────────────────────────────────────────────────────
    is_featured = django_filters.BooleanFilter(field_name="is_featured")

    # ── Amenities (JSON list) ─────────────────────────────────────────────────
    # NOTE: checks if the amenity string appears anywhere in the JSON array
    # Example: /api/properties/?amenity=WiFi
    amenity = django_filters.CharFilter(field_name="amenities", lookup_expr="contains")

    class Meta:
        model = Property
        fields = [
            "city", "district", "type", "status",
            "price_min", "price_max",
            "num_beds", "num_rooms", "num_bathrooms",
            "gender", "university",
            "is_featured", "amenity",
        ]

```
# ────────────────────────End Filters────────────────────────────────────








# ────────────────────────Start Apps────────────────────────────────────
```
"""
properties/apps.py
Registers the auto-geocode signal so it fires on every Property save.
"""
from django.apps import AppConfig


class PropertiesConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name               = "properties"

    def ready(self):
        import properties.signals  # noqa: F401 — registers geocoding signal

```
# ────────────────────────End Apps──────────────────────────────────────






# ────────────────────────Start Model────────────────────────────────────

```
"""
Properties app models.
Handles property listings and their images.

Models:
    - Property      → main listing created by landlords
    - PropertyImage → multiple photos per property
"""


from django.db import models
from django.core.validators import MinValueValidator, MaxValueValidator
from accounts.models import Users

# ──────────────────────────────────────────────────────────────────────────────────────────


class Property(models.Model):
    """
    Main listing model. Created by landlords, browsed by students.

    Shown on: Home page cards, FindRoom page, Property detail page,
              Owner 'My Properties' dashboard.
    """

    # ── Choices ──────────────────────────────────────────────
    PROPERTY_TYPE_CHOICES = [
        ("apartment", "Apartment"),
        ("studio", "Studio"),
        ("room", "Room"),
        ("shared", "Shared Room"),
    ]
    STATUS_CHOICES = [
        ("available", "Available"),
        ("rented", "Rented"),
        ("unavailable", "Unavailable"),
        ("reserved", "Reserved"),
    ]
    TRANSPORT_CHOICES = [
        ("walk", "Walking"),
        ("metro", "Metro"),
        ("transport", "Public Transport"),
    ]
    GENDER_CHOICES = [
        ("male", "Males Only"),
        ("female", "Females Only"),
    ]

    # ── Ownership ────────────────────────────────────────────
    # Only landlords can own properties — enforced at the API layer
    landlord = models.ForeignKey(Users, on_delete=models.CASCADE, related_name="landlord_properties")

    # ── Basic Info ───────────────────────────────────────────
    title         = models.CharField(max_length=255)
    description   = models.TextField(blank=True, null=True)
    property_type = models.CharField(max_length=20, choices=PROPERTY_TYPE_CHOICES)

    # ── Pricing ──────────────────────────────────────────────
    price = models.DecimalField(max_digits=10, decimal_places=2, validators=[MinValueValidator(0)])  

    # ── Location ─────────────────────────────────────────────
    city      = models.CharField(max_length=100)
    district  = models.CharField(max_length=100, blank=True, null=True)
    address   = models.TextField(blank=True, null=True)
    latitude  = models.DecimalField(max_digits=9, decimal_places=6, blank=True, null=True)
    longitude = models.DecimalField(max_digits=9, decimal_places=6, blank=True, null=True)  

    # ── University Proximity ─────────────────────────────────
    # Used in FindRoom university tab filter
    nearby_university      = models.CharField(max_length=255, blank=True, null=True)
    distance_to_university = models.CharField(max_length=50, blank=True, null=True) 
    transport_type         = models.CharField(max_length=20, choices=TRANSPORT_CHOICES, blank=True, null=True)

    # ── Room Details ─────────────────────────────────────────
    num_rooms         = models.IntegerField(default=1)
    num_beds          = models.IntegerField(default=1)  
    num_bathrooms     = models.IntegerField(default=1)
    num_roommates     = models.IntegerField(default=0)  
    floor             = models.IntegerField(blank=True, null=True)
    area_sqm          = models.IntegerField(blank=True, null=True)  
    gender_preference = models.CharField(max_length=10, choices=GENDER_CHOICES)

    # ── Amenities ────────────────────────────────────────────
    # Stored as JSON list e.g. ["WiFi", "AC", "Washing Machine", "Parking"]
    amenities = models.JSONField(default=list, blank=True)

    # ── Stay Duration ────────────────────────────────────────
    # Matches "Length of Stay" filter on FindRoom page
    min_stay_months = models.IntegerField(default=1)
    max_stay_months = models.IntegerField(blank=True, null=True)

    # ── Status & Visibility ──────────────────────────────────
    status      = models.CharField(max_length=20, choices=STATUS_CHOICES, default="available")
    is_featured = models.BooleanField(default=False)  # shown in "Featured Properties" section

    # ── Analytics ────────────────────────────────────────────
    # NOTE: Increment view_count in the detail view each time a user opens a listing
    view_count = models.IntegerField(default=0)  # tracked for Owner dashboard analytics

    # ── Timestamps ───────────────────────────────────────────
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        verbose_name_plural = "Properties"
        ordering = ["-created_at"]  # newest listings first

    def __str__(self):
        return f"{self.title} — {self.landlord.username}"

    # ── Computed Properties ──────────────────────────────────
    @property
    def average_rating(self):
        """Average star rating from all reviews on this property. """
        reviews = self.reviews.all()
        if not reviews.exists():
            return 0
        return round(reviews.aggregate(models.Avg("rating"))["rating__avg"], 1)

    @property
    def review_count(self):
        """Total number of reviews for this property."""
        return self.reviews.count()


# ──────────────────────────────────────────────────────────────────────────────────────────


class PropertyImage(models.Model):
    """
    Multiple photos per property.
    The first image (is_cover=True) is used as the card thumbnail.

    Shown on: Property cards, Property detail photo gallery
    """

    property    = models.ForeignKey(Property, on_delete=models.CASCADE, related_name="images")
    image       = models.ImageField(upload_to="property_images/")
    is_cover    = models.BooleanField(default=False)  # main thumbnail shown on cards
    uploaded_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        ordering = ["-is_cover", "uploaded_at"]  

    def __str__(self):
        return f"Image for {self.property.title} ({'Cover' if self.is_cover else 'Gallery'})"
```

# ────────────────────────End Model──────────────────────────────────────



## services app + google map integration ## 



# ────────────────────────Start URL─────────────────────────────────────

```
"""Services API URL configuration."""
from django.urls import path
from api.services_api.views import (
    NearbyView,
    NearbyUniversityView,
    UniversityListView,
    PropertyDistanceView,
)

urlpatterns = [
    path("nearby/",       NearbyView.as_view(),          name="services-nearby"),
    path("university/",   NearbyUniversityView.as_view(), name="services-university"),
    path("universities/", UniversityListView.as_view(),   name="services-universities-list"),
    path("distance/",     PropertyDistanceView.as_view(), name="services-distance"),
]
        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```
"""
api/services_api/serializers.py

Read-only serializers for nearby place results.
No write serializers needed — NearbyPlace rows are managed internally.
"""

from rest_framework import serializers
from services.models import NearbyPlace
from services.universities import list_universities


class NearbyPlaceSerializer(serializers.ModelSerializer):
    class Meta:
        model  = NearbyPlace
        fields = [
            "id",
            "external_id",
            "name",
            "address",
            "latitude",
            "longitude",
            "distance_m",
            "place_type",
            "rating",
            "open_now",
        ]


class UniversityListSerializer(serializers.Serializer):
    """Returns the list of supported universities."""
    universities = serializers.ListField(child=serializers.CharField())

    def to_representation(self, instance):
        return {"universities": list_universities()}

```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```
"""
api/services_api/views.py

Four endpoints:

1. GET /api/services/nearby/
   ?lat=&lng=&type=&radius=
   → finds amenities near the student's current GPS location

2. GET /api/services/university/
   ?name=&type=&radius=
   → finds amenities near a named Egyptian university

3. GET /api/services/universities/
   → returns all supported university names (public, no auth)

4. GET /api/services/distance/
   ?property_id=&mode=walking
   → returns distance + travel time from a property to its nearby_university
     using Google Distance Matrix API

Cache strategy (for endpoints 1 & 2)
──────────────────────────────────────
Lat/lng rounded to 3 decimal places (~111m buckets).
Fresh = cached_at within CACHE_TTL_HOURS.
Fresh rows → return from DB immediately.
Stale/missing → call Google Places → delete old rows → bulk_create new rows.
"""

import logging
from datetime import timedelta

from django.utils import timezone
from rest_framework.permissions import IsAuthenticated, AllowAny
from rest_framework.response import Response
from rest_framework.views import APIView
from rest_framework import status

from services.models import NearbyPlace
from services.google_maps import fetch_nearby, get_distance_to_university, GoogleMapsError
from services.universities import get_university_coords, list_universities
from properties.models import Property
from .serializers import NearbyPlaceSerializer

logger = logging.getLogger(__name__)

CACHE_TTL_HOURS  = 24
DEFAULT_RADIUS_M = 1500
MAX_RADIUS_M     = 5000
VALID_PLACE_TYPES = [pt[0] for pt in NearbyPlace.PLACE_TYPE_CHOICES]


# ── Helpers ───────────────────────────────────────────────────────────────────

def _round_coord(value: float, decimals: int = 3) -> float:
    return round(value, decimals)


def _get_or_refresh_cache(lat: float, lng: float, place_type: str, radius_m: int) -> list:
    """
    Returns fresh NearbyPlace rows from DB cache.
    Falls back to Google Places API if cache is stale or empty.
    """
    q_lat  = _round_coord(lat)
    q_lng  = _round_coord(lng)
    cutoff = timezone.now() - timedelta(hours=CACHE_TTL_HOURS)

    fresh_qs = NearbyPlace.objects.filter(
        query_lat  = q_lat,
        query_lng  = q_lng,
        place_type = place_type,
        radius_m   = radius_m,
        cached_at__gte = cutoff,
    ).order_by("distance_m")

    if fresh_qs.exists():
        return list(fresh_qs)

    # Cache miss — call Google Places
    raw_places = fetch_nearby(lat, lng, place_type, radius_m)

    # Wipe stale rows for this bucket
    NearbyPlace.objects.filter(
        query_lat  = q_lat,
        query_lng  = q_lng,
        place_type = place_type,
        radius_m   = radius_m,
    ).delete()

    if not raw_places:
        return []

    objs = [
        NearbyPlace(
            query_lat   = q_lat,
            query_lng   = q_lng,
            place_type  = place_type,
            radius_m    = radius_m,
            external_id = p["external_id"],
            name        = p["name"],
            address     = p["address"],
            latitude    = p["latitude"],
            longitude   = p["longitude"],
            distance_m  = p["distance_m"],
            rating      = p["rating"],
            open_now    = p["open_now"],
        )
        for p in raw_places
    ]
    created = NearbyPlace.objects.bulk_create(objs)
    return sorted(created, key=lambda x: x.distance_m)


def _parse_nearby_params(request):
    """Parses and validates ?lat, ?lng, ?type, ?radius."""
    errors = []

    try:
        lat = float(request.query_params.get("lat"))
    except (TypeError, ValueError):
        lat = None
        errors.append("`lat` must be a valid decimal number (e.g. 30.026).")

    try:
        lng = float(request.query_params.get("lng"))
    except (TypeError, ValueError):
        lng = None
        errors.append("`lng` must be a valid decimal number (e.g. 31.213).")

    type_raw = request.query_params.get("type", "supermarket")
    if type_raw not in VALID_PLACE_TYPES:
        errors.append(f"`type` must be one of: {', '.join(VALID_PLACE_TYPES)}.")
        type_raw = None

    try:
        radius_m = min(int(request.query_params.get("radius", DEFAULT_RADIUS_M)), MAX_RADIUS_M)
        if radius_m <= 0:
            raise ValueError
    except (TypeError, ValueError):
        radius_m = DEFAULT_RADIUS_M
        errors.append(f"`radius` must be a positive integer (max {MAX_RADIUS_M}).")

    return lat, lng, type_raw, radius_m, errors


# ── Views ─────────────────────────────────────────────────────────────────────

class NearbyView(APIView):
    """
    GET /api/services/nearby/
    Finds amenities near the student's current GPS location.

    Params:
        lat     (required)
        lng     (required)
        type    (optional, default: supermarket)
        radius  (optional, default: 1500, max: 5000)
    """
    permission_classes = [IsAuthenticated]

    def get(self, request):
        lat, lng, place_type, radius_m, errors = _parse_nearby_params(request)
        if errors:
            return Response({"errors": errors}, status=status.HTTP_400_BAD_REQUEST)

        places = _get_or_refresh_cache(lat, lng, place_type, radius_m)
        serializer = NearbyPlaceSerializer(places, many=True)
        return Response({
            "count":      len(places),
            "place_type": place_type,
            "radius_m":   radius_m,
            "results":    serializer.data,
        })


class NearbyUniversityView(APIView):
    """
    GET /api/services/university/
    Finds amenities near a named Egyptian university.

    Params:
        name    (required)  — must match universities registry
        type    (optional, default: supermarket)
        radius  (optional, default: 1500, max: 5000)
    """
    permission_classes = [IsAuthenticated]

    def get(self, request):
        uni_name   = request.query_params.get("name", "").strip()
        type_raw   = request.query_params.get("type", "supermarket")
        radius_raw = request.query_params.get("radius", str(DEFAULT_RADIUS_M))

        if not uni_name:
            return Response(
                {"error": "`name` is required. Use /api/services/universities/ to see valid names."},
                status=status.HTTP_400_BAD_REQUEST,
            )

        coords = get_university_coords(uni_name)
        if not coords:
            return Response(
                {
                    "error": f"University '{uni_name}' not found.",
                    "hint":  "Use GET /api/services/universities/ to see supported names.",
                },
                status=status.HTTP_404_NOT_FOUND,
            )

        if type_raw not in VALID_PLACE_TYPES:
            return Response(
                {"error": f"`type` must be one of: {', '.join(VALID_PLACE_TYPES)}."},
                status=status.HTTP_400_BAD_REQUEST,
            )

        try:
            radius_m = min(int(radius_raw), MAX_RADIUS_M)
            if radius_m <= 0:
                raise ValueError
        except (TypeError, ValueError):
            radius_m = DEFAULT_RADIUS_M

        lat = float(coords["lat"])
        lng = float(coords["lng"])

        places = _get_or_refresh_cache(lat, lng, type_raw, radius_m)
        serializer = NearbyPlaceSerializer(places, many=True)
        return Response({
            "university": uni_name,
            "city":       coords.get("city", ""),
            "count":      len(places),
            "place_type": type_raw,
            "radius_m":   radius_m,
            "results":    serializer.data,
        })


class UniversityListView(APIView):
    """
    GET /api/services/universities/
    Returns all supported university names for frontend dropdowns.
    Public — no auth required.
    """
    permission_classes = [AllowAny]

    def get(self, request):
        return Response({"universities": list_universities()})


class PropertyDistanceView(APIView):
    """
    GET /api/services/distance/
    Returns real distance and travel time from a property to its nearby_university
    using Google Distance Matrix API.

    Params:
        property_id  (required)  — ID of the property
        mode         (optional)  — walking | transit | driving (default: walking)

    Response:
        {
            "property_id":    1,
            "university":     "Cairo University",
            "distance_text":  "1.2 km",
            "distance_value": 1200,
            "duration_text":  "15 mins",
            "duration_value": 900,
            "mode":           "walking"
        }
    """
    permission_classes = [IsAuthenticated]

    VALID_MODES = ["walking", "transit", "driving"]

    def get(self, request):
        property_id = request.query_params.get("property_id")
        mode        = request.query_params.get("mode", "walking")

        if not property_id:
            return Response(
                {"error": "`property_id` is required."},
                status=status.HTTP_400_BAD_REQUEST,
            )

        if mode not in self.VALID_MODES:
            return Response(
                {"error": f"`mode` must be one of: {', '.join(self.VALID_MODES)}."},
                status=status.HTTP_400_BAD_REQUEST,
            )

        try:
            prop = Property.objects.get(id=property_id)
        except Property.DoesNotExist:
            return Response(
                {"error": "Property not found."},
                status=status.HTTP_404_NOT_FOUND,
            )

        if not prop.latitude or not prop.longitude:
            return Response(
                {"error": "Property has no coordinates. Address may not have been geocoded yet."},
                status=status.HTTP_400_BAD_REQUEST,
            )

        if not prop.nearby_university:
            return Response(
                {"error": "This property has no university set."},
                status=status.HTTP_400_BAD_REQUEST,
            )

        try:
            result = get_distance_to_university(
                origin_lat  = float(prop.latitude),
                origin_lng  = float(prop.longitude),
                destination = prop.nearby_university,
                mode        = mode,
            )
        except GoogleMapsError as exc:
            logger.warning("Distance Matrix error for property %s: %s", property_id, exc)
            return Response(
                {"error": "Could not calculate distance. Please try again."},
                status=status.HTTP_502_BAD_GATEWAY,
            )

        return Response({
            "property_id":    prop.id,
            "university":     prop.nearby_university,
            **result,
        })
```

# ────────────────────────End View──────────────────────────────────────


# ────────────────────────Start Google Maps────────────────────────────────────
```
"""
services/google_maps.py

Unified Google Maps service.
Wraps three Google APIs used across the StudentHub backend:

    1. Geocoding API      → address text → (lat, lng)
    2. Places API         → nearby POIs around a coordinate
    3. Distance Matrix API → distance + travel time between two points

All methods return clean Python dicts and raise GoogleMapsError on failure.
The views/signals that call these methods handle the exception.

Requires GOOGLE_MAPS_API_KEY in settings.py.
"""

import logging
import requests
from django.conf import settings

logger = logging.getLogger(__name__)

GMAPS_BASE = "https://maps.googleapis.com/maps/api"

# Maps our internal place_type to Google Places API type
GOOGLE_PLACE_TYPE_MAP: dict[str, str] = {
    "supermarket":   "supermarket",
    "pharmacy":      "pharmacy",
    "restaurant":    "restaurant",
    "cafe":          "cafe",
    "gym":           "gym",
    "hospital":      "hospital",
    "bank":          "bank",
    "atm":           "atm",
    "bus_station":   "bus_station",
    "metro_station": "subway_station",
    "mosque":        "mosque",
    "library":       "library",
    "laundry":       "laundry",
}

# Maps our internal place_type to Distance Matrix travel mode
TRAVEL_MODE_MAP: dict[str, str] = {
    "walk":      "walking",
    "metro":     "transit",
    "transport": "transit",
}


class GoogleMapsError(Exception):
    """Raised when a Google Maps API call fails."""
    pass


def _api_key() -> str:
    key = getattr(settings, "GOOGLE_MAPS_API_KEY", None)
    if not key:
        raise GoogleMapsError("GOOGLE_MAPS_API_KEY is not set in settings.")
    return key


# ── 1. Geocoding ──────────────────────────────────────────────────────────────

def geocode_address(address: str) -> dict:
    """
    Converts a free-text address to lat/lng.

    Returns:
        {
            "lat":              float,
            "lng":              float,
            "formatted_address": str,
        }

    Raises GoogleMapsError if address cannot be resolved.

    Example:
        geocode_address("15 Tahrir Square, Cairo")
        → {"lat": 30.0444, "lng": 31.2357, "formatted_address": "Tahrir Square, Cairo..."}
    """
    try:
        resp = requests.get(
            f"{GMAPS_BASE}/geocode/json",
            params={"address": address, "key": _api_key(), "region": "eg"},
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
    except requests.RequestException as exc:
        raise GoogleMapsError(f"Geocoding request failed: {exc}") from exc

    if data.get("status") != "OK" or not data.get("results"):
        raise GoogleMapsError(
            f"Geocoding failed for '{address}'. Status: {data.get('status')}"
        )

    result   = data["results"][0]
    location = result["geometry"]["location"]

    return {
        "lat":               location["lat"],
        "lng":               location["lng"],
        "formatted_address": result.get("formatted_address", address),
    }


# ── 2. Places API ─────────────────────────────────────────────────────────────

def fetch_nearby(lat: float, lng: float, place_type: str, radius_m: int = 1500) -> list[dict]:
    """
    Finds POIs near (lat, lng) using Google Places Nearby Search.

    Returns a list of dicts (max 20, sorted by distance):
        {
            "external_id": str,      # Google place_id
            "name":        str,
            "address":     str,
            "latitude":    float,
            "longitude":   float,
            "distance_m":  int,      # straight-line metres from query point
            "rating":      float | None,
            "open_now":    bool | None,
        }

    Returns [] on any error (caller falls back to DB cache).
    """
    google_type = GOOGLE_PLACE_TYPE_MAP.get(place_type)
    if not google_type:
        return []

    try:
        resp = requests.get(
            f"{GMAPS_BASE}/place/nearbysearch/json",
            params={
                "location": f"{lat},{lng}",
                "radius":   radius_m,
                "type":     google_type,
                "key":      _api_key(),
            },
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
    except requests.RequestException as exc:
        logger.warning("Google Places request failed: %s", exc)
        return []

    if data.get("status") not in ("OK", "ZERO_RESULTS"):
        logger.warning("Google Places API status: %s", data.get("status"))
        return []

    results = []
    for place in data.get("results", []):
        loc      = place.get("geometry", {}).get("location", {})
        place_lat = loc.get("lat")
        place_lng = loc.get("lng")
        if place_lat is None or place_lng is None:
            continue

        name = place.get("name", "").strip()
        if not name:
            continue

        open_periods = place.get("opening_hours", {})
        open_now     = open_periods.get("open_now") if open_periods else None
        rating       = place.get("rating")

        results.append({
            "external_id": place.get("place_id", ""),
            "name":        name,
            "address":     place.get("vicinity", ""),
            "latitude":    place_lat,
            "longitude":   place_lng,
            "distance_m":  _haversine_m(lat, lng, place_lat, place_lng),
            "rating":      rating,
            "open_now":    open_now,
        })

    results.sort(key=lambda x: x["distance_m"])
    return results[:20]


# ── 3. Distance Matrix ────────────────────────────────────────────────────────

def get_distance_to_university(
    origin_lat: float,
    origin_lng: float,
    destination: str,
    mode: str = "walking",
) -> dict:
    """
    Calculates distance and travel time from a property to a university.

    Args:
        origin_lat:   property latitude
        origin_lng:   property longitude
        destination:  university name or address string
        mode:         "walking" | "transit" | "driving" (default: walking)

    Returns:
        {
            "distance_text":  "1.2 km",
            "distance_value": 1200,        # metres
            "duration_text":  "15 mins",
            "duration_value": 900,         # seconds
            "mode":           "walking",
        }

    Raises GoogleMapsError if the API call fails or no route is found.
    """
    try:
        resp = requests.get(
            f"{GMAPS_BASE}/distancematrix/json",
            params={
                "origins":      f"{origin_lat},{origin_lng}",
                "destinations": destination,
                "mode":         mode,
                "key":          _api_key(),
                "region":       "eg",
                "language":     "en",
            },
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
    except requests.RequestException as exc:
        raise GoogleMapsError(f"Distance Matrix request failed: {exc}") from exc

    if data.get("status") != "OK":
        raise GoogleMapsError(
            f"Distance Matrix failed. Status: {data.get('status')}"
        )

    try:
        element = data["rows"][0]["elements"][0]
    except (IndexError, KeyError) as exc:
        raise GoogleMapsError("Unexpected Distance Matrix response structure.") from exc

    if element.get("status") != "OK":
        raise GoogleMapsError(
            f"No route found. Element status: {element.get('status')}"
        )

    return {
        "distance_text":  element["distance"]["text"],
        "distance_value": element["distance"]["value"],
        "duration_text":  element["duration"]["text"],
        "duration_value": element["duration"]["value"],
        "mode":           mode,
    }


# ── Utility ───────────────────────────────────────────────────────────────────

def _haversine_m(lat1: float, lng1: float, lat2: float, lng2: float) -> int:
    """Straight-line distance in metres between two GPS coordinates."""
    import math
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lng2 - lng1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return int(R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))

```
# ────────────────────────End Google Maps──────────────────────────────────────


# ────────────────────────Start university────────────────────────────────────
```

"""
services/universities.py

Coordinate registry for Egyptian universities.
Used for:
  1. NearbyUniversityView  → find amenities near a university
  2. Distance Matrix calls → resolve university name to a searchable string

Add more entries as the platform expands.
"""

EGYPTIAN_UNIVERSITIES: dict[str, dict] = {
    # ── Cairo ──────────────────────────────────────────────────────
    "Cairo University":                          {"lat": 30.0262,  "lng": 31.2132,  "city": "Giza"},
    "Ain Shams University":                      {"lat": 30.0730,  "lng": 31.2798,  "city": "Cairo"},
    "Helwan University":                         {"lat": 29.8497,  "lng": 31.3238,  "city": "Cairo"},
    "Misr University for Science and Technology":{"lat": 29.8814,  "lng": 31.3506,  "city": "Giza"},
    "German University in Cairo":                {"lat": 29.9878,  "lng": 31.4414,  "city": "Cairo"},
    "American University in Cairo":              {"lat": 30.0220,  "lng": 31.4999,  "city": "Cairo"},
    "Future University in Egypt":                {"lat": 30.0281,  "lng": 31.4762,  "city": "Cairo"},
    "Modern Sciences and Arts University":       {"lat": 30.0617,  "lng": 31.2281,  "city": "Giza"},
    "Nile University":                           {"lat": 30.0618,  "lng": 30.8426,  "city": "Giza"},
    "October 6 University":                      {"lat": 29.9350,  "lng": 30.9296,  "city": "6th of October"},
    "Badr University in Cairo":                  {"lat": 30.1290,  "lng": 31.7439,  "city": "Cairo"},

    # ── Alexandria ────────────────────────────────────────────────
    "Alexandria University":                     {"lat": 31.2001,  "lng": 29.9187,  "city": "Alexandria"},
    "Arab Academy for Science and Technology":   {"lat": 31.1842,  "lng": 29.9370,  "city": "Alexandria"},
    "Pharos University in Alexandria":           {"lat": 31.1697,  "lng": 29.9663,  "city": "Alexandria"},

    # ── Delta / Canal ─────────────────────────────────────────────
    "Mansoura University":                       {"lat": 31.0409,  "lng": 31.3785,  "city": "Mansoura"},
    "Tanta University":                          {"lat": 30.7969,  "lng": 31.0039,  "city": "Tanta"},
    "Zagazig University":                        {"lat": 30.5855,  "lng": 31.5027,  "city": "Zagazig"},
    "Suez Canal University":                     {"lat": 30.6057,  "lng": 32.2694,  "city": "Ismailia"},
    "Port Said University":                      {"lat": 31.2544,  "lng": 32.2940,  "city": "Port Said"},
    "Benha University":                          {"lat": 30.4628,  "lng": 31.1856,  "city": "Benha"},
    "Kafr El Sheikh University":                 {"lat": 31.1063,  "lng": 30.9388,  "city": "Kafr El Sheikh"},
    "Damietta University":                       {"lat": 31.4218,  "lng": 31.8131,  "city": "Damietta"},

    # ── Upper Egypt ───────────────────────────────────────────────
    "Assiut University":                         {"lat": 27.1810,  "lng": 31.1655,  "city": "Assiut"},
    "Sohag University":                          {"lat": 26.5569,  "lng": 31.6948,  "city": "Sohag"},
    "South Valley University":                   {"lat": 25.6872,  "lng": 32.6439,  "city": "Qena"},
    "Aswan University":                          {"lat": 24.0889,  "lng": 32.8998,  "city": "Aswan"},
    "Luxor University":                          {"lat": 25.6872,  "lng": 32.6439,  "city": "Luxor"},

    # ── Sinai / Red Sea ───────────────────────────────────────────
    "Sinai University":                          {"lat": 30.9753,  "lng": 33.7959,  "city": "Arish"},
    "Galala University":                         {"lat": 29.4938,  "lng": 32.1197,  "city": "Ain Sokhna"},

    # ── New cities ────────────────────────────────────────────────
    "Zewail City of Science and Technology":     {"lat": 29.9338,  "lng": 30.9323,  "city": "6th of October"},
    "New Mansoura University":                   {"lat": 31.5173,  "lng": 32.0256,  "city": "New Mansoura"},
}


def get_university_coords(name: str) -> dict | None:
    """
    Case-insensitive lookup.
    Returns {"lat": float, "lng": float, "city": str} or None.
    Falls back to partial match if exact match not found.
    """
    name_lower = name.strip().lower()
    for uni_name, coords in EGYPTIAN_UNIVERSITIES.items():
        if uni_name.lower() == name_lower:
            return coords
    for uni_name, coords in EGYPTIAN_UNIVERSITIES.items():
        if name_lower in uni_name.lower():
            return coords
    return None


def list_universities() -> list[str]:
    return sorted(EGYPTIAN_UNIVERSITIES.keys())
```
# ────────────────────────End university──────────────────────────────────────



# ────────────────────────Start Signals────────────────────────────────────
```


```
# ────────────────────────End Signals──────────────────────────────────────





# ────────────────────────Start Apps────────────────────────────────────
```


```
# ────────────────────────End Apps──────────────────────────────────────






# ────────────────────────Start Model────────────────────────────────────

```
"""
services/models.py

Cache layer for Google Places API results.
Rows are refreshed after CACHE_TTL_HOURS (set in views.py).
"""

from django.db import models


class NearbyPlace(models.Model):

    PLACE_TYPE_CHOICES = [
        ("supermarket",   "Supermarket"),
        ("pharmacy",      "Pharmacy"),
        ("restaurant",    "Restaurant"),
        ("cafe",          "Café"),
        ("gym",           "Gym"),
        ("hospital",      "Hospital"),
        ("bank",          "Bank"),
        ("atm",           "ATM"),
        ("bus_station",   "Bus Station"),
        ("metro_station", "Metro Station"),
        ("mosque",        "Mosque"),
        ("library",       "Library"),
        ("laundry",       "Laundry"),
        ("other",         "Other"),
    ]

    # ── Query context (cache bucket) ──────────────────────────
    query_lat  = models.DecimalField(max_digits=9, decimal_places=6)
    query_lng  = models.DecimalField(max_digits=9, decimal_places=6)
    place_type = models.CharField(max_length=30, choices=PLACE_TYPE_CHOICES)
    radius_m   = models.PositiveIntegerField(default=1500)

    # ── Place details ─────────────────────────────────────────
    external_id = models.CharField(max_length=255, blank=True)  # Google place_id
    name        = models.CharField(max_length=255)
    address     = models.CharField(max_length=500, blank=True)
    latitude    = models.DecimalField(max_digits=9, decimal_places=6)
    longitude   = models.DecimalField(max_digits=9, decimal_places=6)
    distance_m  = models.PositiveIntegerField(default=0)
    rating      = models.DecimalField(max_digits=3, decimal_places=1, null=True, blank=True)
    open_now    = models.BooleanField(null=True, blank=True)

    cached_at = models.DateTimeField(auto_now=True)

    class Meta:
        ordering = ["distance_m"]
        indexes  = [
            models.Index(fields=["query_lat", "query_lng", "place_type"]),
        ]

    def __str__(self):
        return f"{self.name} ({self.place_type}) — {self.distance_m}m"
```

# ────────────────────────End Model──────────────────────────────────────



## payment overview ##


# ────────────────────────Start URL─────────────────────────────────────

```
from django.urls import path
from .views import (
    InitiateDepositView,
    PayRemainingOnlineView,
    MarkRemainingOfflineView,
    MyPaymentsView,
    PaymobWebhookView,
)

urlpatterns = [
    path("deposit/",           InitiateDepositView.as_view(),     name="pay-deposit"),
    path("remaining/online/",  PayRemainingOnlineView.as_view(),  name="pay-remaining-online"),
    path("remaining/offline/", MarkRemainingOfflineView.as_view(),name="pay-remaining-offline"),
    path("my/",                MyPaymentsView.as_view(),          name="my-payments"),
    path("webhook/",           PaymobWebhookView.as_view(),       name="paymob-webhook"),
]
        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```


from rest_framework import serializers
from payments.models import Payment
from bookings.models import Booking 




class PaymentSerializer(serializers.ModelSerializer):
    """Read-only — used to display payment info."""
    class Meta:
        model = Payment
        fields = [
            "id",
            "booking",
            "payment_type",
            "payment_method",
            "amount_cents",
            "status",
            "paymob_order_id",
            "transaction_id",
            "failure_reason",
            "paid_at",
            "created_at",
        ]

        read_only_fields = fields


#──────────Deposit────────────────────────────────────────────────────────────────────────────────────────────

class InitiateDepositSerializer(serializers.Serializer):
    """Validates deposit initiation request."""
    booking_id = serializers.IntegerField()
    phone      = serializers.CharField(max_length=20 , required=False, default="NA")

    def validate_booking_id(self, value):
        request = self.context.get("request")

        try:
            booking = Booking.objects.get(
                id     = value,
                tenant = request.user,
                status = "pending_payment",
            )
        except Booking.DoesNotExist:
            raise serializers.ValidationError("Booking not found or not in pending_payment status.")
        

        if booking.is_expired:
            booking.status = "expired"
            booking.save()
            raise serializers.ValidationError("Booking has expired. Please create a new booking.")
        
        # Check duplicate pending deposit
        if booking.payments.filter(
            payment_type = Payment.PaymentType.DEPOSIT,
            status       = Payment.Status.PENDING,
        ).exists():
            raise serializers.ValidationError("A deposit payment is already in progress.")
        
        return value


#──────────Remaining────────────────────────────────────────────────────────────────────────────────────────────

class PayRemainingSerializer(serializers.Serializer):
    """Validates remaining payment initiation request."""
    booking_id = serializers.IntegerField()
    phone      = serializers.CharField(max_length=20, required=False, default="NA")

    def validate_booking_id(self, value):
        request = self.context.get("request")

        try:
            booking = Booking.objects.get(
                id     = value,
                tenant = request.user,
                status = "confirmed",
            )
        except Booking.DoesNotExist:
            raise serializers.ValidationError(
                "Booking not found or not yet confirmed."
            )

        # Prevent duplicate remaining payment
        if booking.payments.filter(
            payment_type = Payment.PaymentType.REMAINING,
            status__in   = [Payment.Status.PENDING, Payment.Status.COMPLETED],
        ).exists():
            raise serializers.ValidationError(
                "Remaining payment already exists for this booking."
            )

        return value


#──────────Offline-Pay────────────────────────────────────────────────────────────────────────────────────────────

class MarkOfflineSerializer(serializers.Serializer):
    """Validates landlord marking remaining payment as cash."""
    booking_id = serializers.IntegerField()

    def validate_booking_id(self, value):
        request = self.context.get("request")

        try:
            booking = Booking.objects.get(
                id                    = value,
                property__landlord    = request.user,
                status                = "confirmed",
            )
        except Booking.DoesNotExist:
            raise serializers.ValidationError(
                "Booking not found or you don't own this property."
            )

        if booking.payments.filter(
            payment_type = Payment.PaymentType.REMAINING,
            status__in   = [Payment.Status.PENDING, Payment.Status.COMPLETED],
        ).exists():
            raise serializers.ValidationError(
                "Remaining payment already recorded for this booking."
            )

        return value
```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```

import json
import hmac
import hashlib

from django.db import transaction
from django.utils import timezone
from django.conf import settings
from django.views import View
from django.http import HttpResponse
from django.utils.decorators import method_decorator
from django.views.decorators.csrf import csrf_exempt

from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework.permissions import IsAuthenticated
from rest_framework import status

from bookings.models import Booking
from payments.models import Payment
from payments.paymob import PaymobService
from .serializers import (
    PaymentSerializer,
    InitiateDepositSerializer,
    PayRemainingSerializer,
    MarkOfflineSerializer,
)


class InitiateDepositView(APIView):
    """
    Step 1 of payment flow.
    Student initiates deposit
    
    the overview flow of the whole payment proccess is :
        Create Intention
            ↓
        Redirect User
            ↓
        Payment Happens
            ↓
        Webhook Arrives
            ↓
        Verify HMAC
            ↓
        Update Database
    """
    permission_classes = [IsAuthenticated]

    def post(self, request):
        serializer = InitiateDepositSerializer(
            data=request.data, context={"request": request}
        )
        serializer.is_valid(raise_exception=True)

        booking = Booking.objects.get(id=serializer.validated_data["booking_id"])

        billing_data = {
            "first_name":      request.user.first_name or "NA",
            "last_name":       request.user.last_name  or "NA",
            "email":           request.user.email,
            "phone_number":    serializer.validated_data["phone"],
            "apartment":       "NA",
            "floor":           "NA",
            "street":          "NA",
            "building":        "NA",
            "shipping_method": "NA",
            "postal_code":     "NA",
            "city":            "NA",
            "country":         "EG",
            "state":           "NA",
        }

        try:
                     #Create Payment Intention
            result = PaymobService.initiate_payment(
                amount_cents = booking.deposit_amount_cents,
                billing_data = billing_data,
            )

            Payment.objects.create(
                booking         = booking,
                payment_type    = Payment.PaymentType.DEPOSIT,
                payment_method  = Payment.PaymentMethod.ONLINE,
                amount_cents    = booking.deposit_amount_cents,
                paymob_order_id = str(result["order_id"]),
                status          = Payment.Status.PENDING,
            )

            return Response(result)

        except Exception as e:
            return Response({"error": str(e)}, status=status.HTTP_502_BAD_GATEWAY)

#──────────────────────────────────────────────────────────────────────────────────────────────────────

class PayRemainingOnlineView(APIView):
    """Student pays remaining amount online via Paymob."""
    permission_classes = [IsAuthenticated]

    def post(self, request):
        serializer = PayRemainingSerializer(
            data=request.data, context={"request": request}
        )
        serializer.is_valid(raise_exception=True)

        booking = Booking.objects.get(id=serializer.validated_data["booking_id"])

        billing_data = {
            "first_name":      request.user.first_name or "NA",
            "last_name":       request.user.last_name  or "NA",
            "email":           request.user.email,
            "phone_number":    serializer.validated_data["phone"],
            "apartment":       "NA",
            "floor":           "NA",
            "street":          "NA",
            "building":        "NA",
            "shipping_method": "NA",
            "postal_code":     "NA",
            "city":            "NA",
            "country":         "EG",
            "state":           "NA",
        }

        try:
            result = PaymobService.initiate_payment(
                amount_cents = booking.remaining_amount_cents,
                billing_data = billing_data,
            )

            Payment.objects.create(
                booking         = booking,
                payment_type    = Payment.PaymentType.REMAINING,
                payment_method  = Payment.PaymentMethod.ONLINE,
                amount_cents    = booking.remaining_amount_cents,
                paymob_order_id = str(result["order_id"]),
                status          = Payment.Status.PENDING,
            )

            return Response(result)

        except Exception as e:
            return Response({"error": str(e)}, status=status.HTTP_502_BAD_GATEWAY)


#──────────────────────────────────────────────────────────────────────────────────────────────────────

class MarkRemainingOfflineView(APIView):
    """Landlord marks remaining amount as paid in cash."""
    permission_classes = [IsAuthenticated]

    def post(self, request):
        serializer = MarkOfflineSerializer(
            data=request.data, context={"request": request}
        )
        serializer.is_valid(raise_exception=True)

        booking = Booking.objects.get(id=serializer.validated_data["booking_id"])

        with transaction.atomic():
            Payment.objects.create(
                booking        = booking,
                payment_type   = Payment.PaymentType.REMAINING,
                payment_method = Payment.PaymentMethod.OFFLINE,
                amount_cents   = booking.remaining_amount_cents,
                status         = Payment.Status.COMPLETED,
                paid_at        = timezone.now(),
            )

            booking.status = "completed"
            booking.save()

        return Response({"message": "Cash payment recorded. Booking marked as completed."})


#──────────────────────────────────────────────────────────────────────────────────────────────────────


class MyPaymentsView(APIView):
    """Student views all their payments."""
    permission_classes = [IsAuthenticated]

    def get(self, request):
        payments = Payment.objects.filter(
            booking__tenant=request.user
        ).select_related("booking")

        serializer = PaymentSerializer(payments, many=True)
        return Response(serializer.data)


#──────────────────────────────────────────────────────────────────────────────────────────────────────

@method_decorator(csrf_exempt, name="dispatch")
class PaymobWebhookView(View):
    """
    Paymob calls this after every transaction.
    Flow: verify HMAC → update Payment → update Booking.

    the idea of HMAC is to make new HMAC based on data i have and 
    make sure that it match the received hmac  
    """

    def post(self, request):
        try:
            data = json.loads(request.body)
        except Exception:
            return HttpResponse(status=400)

        received_hmac = request.GET.get("hmac", "")
        # Step 1: verify this is really from Paymob
        if not self._verify_hmac(data, received_hmac):
            return HttpResponse("Invalid HMAC", status=403)

        obj            = data.get("obj", {})
        order_id       = str(obj.get("order", {}).get("id", ""))
        success        = obj.get("success", False)
        transaction_id = str(obj.get("id", ""))

        # Step 2: find the payment record
        try:
            payment = Payment.objects.select_related("booking__property").get(
                paymob_order_id=order_id
            )
        except Payment.DoesNotExist:
            return HttpResponse(status=404)

        # Step 3: idempotency — ignore if already processed
        if payment.status == Payment.Status.COMPLETED:
            return HttpResponse(status=200)

        # Step 4: update payment + booking atomically
        with transaction.atomic():
            if success:
                payment.status         = Payment.Status.COMPLETED
                payment.transaction_id = transaction_id
                payment.raw_response   = obj
                payment.paid_at        = timezone.now()
                payment.save()

                booking = payment.booking
                if payment.payment_type == Payment.PaymentType.DEPOSIT:
                    booking.status = "deposit_paid"
                elif payment.payment_type == Payment.PaymentType.REMAINING:
                    booking.status = "completed"
                booking.save()

            else:
                payment.status         = Payment.Status.FAILED
                payment.failure_reason = obj.get("data", {}).get("message", "")
                payment.raw_response   = obj
                payment.save()

        return HttpResponse(status=200)

    def _verify_hmac(self, data ,received_hmac):
        """Verify Paymob HMAC signature."""
        obj           = data.get("obj", {})

        def fmt(value):
            """Convert Python booleans to lowercase — Paymob expects true/false not True/False"""
            if isinstance(value, bool):
                return "true" if value else "false"
            if value is None:
                return ""
            return str(value)
        fields = [
            fmt(obj.get("amount_cents",           "")),
            fmt(obj.get("created_at",             "")),
            fmt(obj.get("currency",               "")),
            fmt(obj.get("error_occured",          "")),
            fmt(obj.get("has_parent_transaction", "")),
            fmt(obj.get("id",                     "")),
            fmt(obj.get("integration_id",         "")),
            fmt(obj.get("is_3d_secure",           "")),
            fmt(obj.get("is_auth",                "")),
            fmt(obj.get("is_capture",             "")),
            fmt(obj.get("is_refunded",            "")),
            fmt(obj.get("is_standalone_payment",  "")),
            fmt(obj.get("is_voided",              "")),
            fmt(obj.get("order", {}).get("id",    "")),
            fmt(obj.get("owner",                  "")),
            fmt(obj.get("pending",                "")),
            fmt(obj.get("source_data", {}).get("pan",      "")),
            fmt(obj.get("source_data", {}).get("sub_type", "")),
            fmt(obj.get("source_data", {}).get("type",     "")),
            fmt(obj.get("success",                "")),
        ]


        expected = hmac.new(
            settings.PAYMOB_HMAC_SECRET.encode(),
            "".join(fields).encode(),
            hashlib.sha512
        ).hexdigest()

        
        return hmac.compare_digest(expected, received_hmac)
```

# ────────────────────────End View──────────────────────────────────────





# ────────────────────────Start Signals────────────────────────────────────
```


```
# ────────────────────────End Signals──────────────────────────────────────





# ────────────────────────Start Apps────────────────────────────────────
```


```
# ────────────────────────End Apps──────────────────────────────────────






# ────────────────────────Start Model────────────────────────────────────

```
from django.db import models
from bookings.models import Booking



class Payment(models.Model):


    class Status(models.TextChoices):
        PENDING   = "pending" , "Pending"
        COMPLETED = "completed" , "Completed"
        FAILED    = "failed" , "Failed"
        REFUNDED  = "refunded" , "Refunded"

    class PaymentType(models.TextChoices):
        DEPOSIT   = "deposit" , "Deposit"
        REMAINING = "remaining" , "Remaining"

    class PaymentMethod(models.TextChoices):
        ONLINE  = "online",  "Online (Paymob)"
        OFFLINE = "offline", "Offline (Cash)"


    #────────────────────────────────────────────────────────────────────────────────────────
    booking = models.ForeignKey( Booking , on_delete=models.CASCADE ,related_name="payments" )


    #──────Transaction Info──────────────────────────────────────────────────────────────────────────────────
    payment_type    = models.CharField(max_length=20 , choices=PaymentType.choices , default=PaymentType.DEPOSIT)
    payment_method  = models.CharField(max_length=20 , choices=PaymentMethod.choices , default=PaymentMethod.ONLINE )
    amount_cents    = models.PositiveIntegerField()
    status          = models.CharField(max_length=20 ,choices=Status.choices ,default=Status.PENDING)

    #──────Paymob fields──────────────────────────────────────────────────────────────────────────────────
    paymob_order_id = models.CharField(max_length=100, blank=True, null=True)
    transaction_id  = models.CharField(max_length=100, blank=True, null=True, unique=True)
    raw_response   = models.JSONField(default=dict, blank=True)
    failure_reason = models.TextField(blank=True, null=True)
    paid_at        = models.DateTimeField(blank=True, null=True)

    #───────Audit─────────────────────────────────────────────────────────────────────────────────
    created_at     = models.DateTimeField(auto_now_add=True)
    updated_at     = models.DateTimeField(auto_now=True)

    # Add this Meta class to your Payment model
    class Meta:
        constraints = [
            models.UniqueConstraint(
                fields=["booking", "payment_type"],
                condition=models.Q(status="completed"),
                name="unique_completed_payment_type_per_booking"
            )
        ]
        
    def __str__(self):
        return f"{self.payment_type} | {self.booking} | {self.status}"
```

# ────────────────────────End Model──────────────────────────────────────




# ────────────────────────Paymob──────────────────────────────────────

```
import requests
from django.conf import settings

PAYMOB_BASE_URL = "https://accept.paymob.com/api"


class PaymobService:

    @staticmethod
    def get_auth_token():
        """
        Step 1: Authenticate with Paymob.
        Returns auth_token (valid ~1 hour).
        """
        response = requests.post(
            f"{PAYMOB_BASE_URL}/auth/tokens",
            json={"api_key": settings.PAYMOB_API_KEY}
        )
        response.raise_for_status()
        return response.json()["token"]

    @staticmethod
    def create_order(auth_token, amount_cents):
        """
        Step 2: Register order in Paymob.
        Returns order_id.
        """
        response = requests.post(
            f"{PAYMOB_BASE_URL}/ecommerce/orders",
            json={
                "auth_token":      auth_token,
                "delivery_needed": False,
                "amount_cents":    amount_cents,
                "currency":        "EGP",
                "items":           [],
            }
        )
        response.raise_for_status()
        return response.json()["id"]

    @staticmethod
    def get_payment_key(auth_token, order_id, amount_cents, billing_data):
        """
        Step 3: Get short-lived payment key.
        Returns payment_key token used in iframe URL.
        """
        response = requests.post(
            f"{PAYMOB_BASE_URL}/acceptance/payment_keys",
            json={
                "auth_token":    auth_token,
                "amount_cents":  amount_cents,
                "expiration":    3600,
                "order_id":      order_id,
                "billing_data":  billing_data,
                "currency":      "EGP",
                "integration_id": settings.PAYMOB_INTEGRATION_ID,
            }
        )
        response.raise_for_status()
        return response.json()["token"]

    @classmethod
    def initiate_payment(cls, amount_cents, billing_data):
        """
        Runs all 3 steps together.
        Returns iframe_url, order_id, payment_key.
        """
        auth_token  = cls.get_auth_token()
        order_id    = cls.create_order(auth_token, amount_cents)
        payment_key = cls.get_payment_key(auth_token, order_id, amount_cents, billing_data)

        return {
            "iframe_url":  f"https://accept.paymob.com/api/acceptance/iframes/{settings.PAYMOB_IFRAME_ID}?payment_token={payment_key}",
            "order_id":    order_id,
            "payment_key": payment_key,
        }
```
# ────────────────────────End paymob──────────────────────────────────────




## review app ##


# ────────────────────────Start URL─────────────────────────────────────

```
"""Reviews API URL configuration."""
from django.urls import path
from api.reviews_api.views import PropertyReviewListView, UserReviewListView

urlpatterns = [
    # ── Property Reviews ──────────────────────────────────────────────────────
    path("property/<int:property_id>/", PropertyReviewListView.as_view(), name="property-reviews"),

    # ── User Reviews ──────────────────────────────────────────────────────────
    path("user/<int:user_id>/", UserReviewListView.as_view(), name="user-reviews"),
]

        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```
"""
Reviews API serializers.

Serializers:
    - ReviewSerializer              → GET responses (full read)
    - PropertyReviewCreateSerializer → POST /api/reviews/property/<id>/
    - UserReviewCreateSerializer     → POST /api/reviews/user/<id>/
"""

from rest_framework import serializers
from reviews.models import Review
from bookings.models import Booking

class ReviewSerializer(serializers.ModelSerializer):
    """
    Full read serializer. Used in all GET responses.
    Shows reviewer username so the frontend can display the author line.
    """

    reviewer_username    = serializers.CharField(source="reviewer.username", read_only=True)
    reviewer_picture     = serializers.ImageField(source="reviewer.profile_picture", read_only=True)

    class Meta:
        model  = Review
        fields = [
            "id",
            "reviewer", "reviewer_username", "reviewer_picture",
            "reviewer_role",
            "property", "reviewed_user",
            "rating", "comment",
            "created_at",
        ]
        read_only_fields = ["id", "reviewer", "created_at"]


class PropertyReviewCreateSerializer(serializers.ModelSerializer):
    """
    Write serializer for reviewing a property.
    property is injected from the URL — not accepted from the request body.
    reviewer is injected from request.user.

    Rules:
      - Only students can review properties (enforced in view via IsStudent)
      - One review per student per property
      - reviewer_role for property reviews is restricted to landlord (i.e. they are reviewing as a past tenant)
        — actually the role describes the reviewer's relationship, so we limit to sensible choices.
    """
    booking_id = serializers.IntegerField(write_only=True)
    class Meta:
        model  = Review
        fields = ["rating", "comment", "reviewer_role","booking_id"]

    
    def validate_booking_id(self, value):
        request = self.context.get("request")
        try:
            # Ensure booking exists AND belongs to the requesting student
            booking = Booking.objects.get(id=value, tenant=request.user)
        except Booking.DoesNotExist:
            raise serializers.ValidationError("Booking not found or you are not the tenant.")
        
        # Students can only review after the stay is confirmed or completed
        if booking.status not in ["confirmed", "completed"]:
            raise serializers.ValidationError("You can only review a property after your booking is confirmed or completed.")
        
        # Prevent duplicate reviews for the exact same booking
        if Review.objects.filter(booking=booking).exists():
            raise serializers.ValidationError("You have already submitted a review for this booking.")
        
        return value

    def validate_reviewer_role(self, value):
        # For property reviews, the student is reviewing as a past tenant.
        # Only these roles make sense in that context.
        allowed = ["roommate", "neighbor"]
        # NOTE: reviewer_role on a property review describes the reviewer's
        # living arrangement context — keeping all four valid for flexibility.
        allowed = ["landlord", "roommate", "classmate", "neighbor"]
        if value not in allowed:
            raise serializers.ValidationError(
                f"reviewer_role must be one of: {', '.join(allowed)}."
            )
        return value

    def validate(self, data):
        reviewer = self.context["request"].user
        property_obj = self.context["property"]

        # One review per student per property
        if Review.objects.filter(reviewer=reviewer, property=property_obj).exists():
            raise serializers.ValidationError("You have already reviewed this property.")

        return data

    def create(self, validated_data):
        booking_id = validated_data.pop("booking_id")
        booking = Booking.objects.get(id=booking_id)
        
        # Auto-inject fields securely
        validated_data["booking"] = booking
        validated_data["property"] = booking.property
        validated_data["reviewer"] = self.context["request"].user
        
        return Review.objects.create(**validated_data)


class UserReviewCreateSerializer(serializers.ModelSerializer):
    """
    Write serializer for reviewing another user.
    reviewed_user is injected from the URL — not accepted from the request body.
    reviewer is injected from request.user.

    Rules:
      - Cannot review yourself
      - One review per reviewer per reviewed_user
    """

    class Meta:
        model  = Review
        fields = ["rating", "comment", "reviewer_role"]

    def validate(self, data):
        reviewer      = self.context["request"].user
        reviewed_user = self.context["reviewed_user"]

        # Prevent self-reviews
        if reviewer == reviewed_user:
            raise serializers.ValidationError("You cannot review yourself.")

        # One review per reviewer per user
        if Review.objects.filter(reviewer=reviewer, reviewed_user=reviewed_user).exists():
            raise serializers.ValidationError("You have already reviewed this user.")

        return data

    def create(self, validated_data):
        # reviewer and reviewed_user injected from the view
        return Review.objects.create(**validated_data)

```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```
"""
Reviews API views.

Endpoints:
    GET    /api/reviews/property/<id>/  → PropertyReviewListView   (anyone authenticated)
    POST   /api/reviews/property/<id>/  → PropertyReviewListView   (student only)
    GET    /api/reviews/user/<id>/      → UserReviewListView       (anyone authenticated)
    POST   /api/reviews/user/<id>/      → UserReviewListView       (authenticated, not self)
"""

from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from rest_framework.permissions import IsAuthenticated , AllowAny

from reviews.models import Review
from properties.models import Property
from accounts.models import Users
from bookings.models import Booking
from api.reviews_api.serializers import (
    ReviewSerializer,
    PropertyReviewCreateSerializer,
    UserReviewCreateSerializer,
)
from api.accounts_api.permissions import IsStudent


class PropertyReviewListView(APIView):
    """
    GET  /api/reviews/property/<id>/ → list all reviews for a property
    POST /api/reviews/property/<id>/ → student submits a review for a property
    """

    def get_permissions(self):
        """GET is open to anyone authenticated. POST is students only."""
        if self.request.method == "POST":
            return [IsStudent()]
        return [AllowAny()]

    def get(self, request, property_id):
        try:
            prop = Property.objects.get(id=property_id)
        except Property.DoesNotExist:
            return Response({"error": "Property not found."}, status=status.HTTP_404_NOT_FOUND)

        reviews = Review.objects.filter(property=prop).select_related("reviewer")
        serializer = ReviewSerializer(reviews, many=True, context={"request": request})

        return Response({
            "property_id": prop.id,
            "average_rating": prop.average_rating,  # ADD THIS
            "review_count": prop.review_count,      # ADD THIS
            "reviews": serializer.data
        }, status=status.HTTP_200_OK)

    def post(self, request, property_id):
        try:
            prop = Property.objects.get(id=property_id)
        except Property.DoesNotExist:
            return Response({"error": "Property not found."}, status=status.HTTP_404_NOT_FOUND)

        serializer = PropertyReviewCreateSerializer(
            data=request.data,
            context={"request": request, "property": prop},
        )
        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        booking_id = request.data.get("booking_id")
        if booking_id:
            try:
                booking = Booking.objects.get(id=booking_id)
                if booking.property.id != prop.id:
                    return Response({"error": "This booking is not for the specified property."}, status=status.HTTP_400_BAD_REQUEST)
            except Booking.DoesNotExist:
                pass # Handled by serializer validation

        review = serializer.save()
        return Response(ReviewSerializer(review).data, status=status.HTTP_201_CREATED)


class UserReviewListView(APIView):
    """
    GET  /api/reviews/user/<id>/ → list all reviews for a user
    POST /api/reviews/user/<id>/ → authenticated user submits a review for another user
    """
    permission_classes = [IsAuthenticated]

    def get(self, request, user_id):
        try:
            user = Users.objects.get(id=user_id)
        except Users.DoesNotExist:
            return Response({"error": "User not found."}, status=status.HTTP_404_NOT_FOUND)

        reviews = Review.objects.filter(reviewed_user=user).select_related("reviewer")
        serializer = ReviewSerializer(reviews, many=True, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)

    def post(self, request, user_id):
        try:
            reviewed_user = Users.objects.get(id=user_id)
        except Users.DoesNotExist:
            return Response({"error": "User not found."}, status=status.HTTP_404_NOT_FOUND)

        serializer = UserReviewCreateSerializer(
            data=request.data,
            context={"request": request, "reviewed_user": reviewed_user},
        )
        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        review = serializer.save(reviewer=request.user, reviewed_user=reviewed_user)
        return Response(ReviewSerializer(review).data, status=status.HTTP_201_CREATED)

```

# ────────────────────────End View──────────────────────────────────────





# ────────────────────────Start Signals────────────────────────────────────
```


```
# ────────────────────────End Signals──────────────────────────────────────





# ────────────────────────Start Apps────────────────────────────────────
```


```
# ────────────────────────End Apps──────────────────────────────────────






# ────────────────────────Start Model────────────────────────────────────

```
"""
Reviews app models.
Handles star ratings and written reviews for both properties and users.

Models:
    - Review → a rating left on a property OR on another user
"""

from django.db import models
from django.core.validators import MinValueValidator, MaxValueValidator
from accounts.models import Users
from properties.models import Property
from bookings.models import Booking 


class Review(models.Model):
    """
    Unified review model for both property reviews and user reviews.

    A review targets EITHER a property or a user — never both at once.
    The target type is determined by which FK is set (property vs reviewed_user).

    reviewer_role tells us the relationship between the reviewer and the subject:
        - property reviews: reviewer is always a student (past tenant)
        - user reviews:     landlord ↔ student, or student ↔ student (roommate, classmate, neighbor)
    """

    REVIEWER_ROLE_CHOICES = [
        ("landlord",  "Landlord"),
        ("roommate",  "Roommate"),
        ("classmate", "Classmate"),
        ("neighbor",  "Neighbor"),
    ]

    # ── Who is reviewing ──────────────────────────────────────────────────────
    reviewer = models.ForeignKey(
        Users,
        on_delete=models.CASCADE,
        related_name="reviews_given",
    )
    reviewer_role = models.CharField(max_length=20, choices=REVIEWER_ROLE_CHOICES)

    # ── What is being reviewed (exactly one must be set) ──────────────────────
    # Property review — student reviews a property they stayed at
    property = models.ForeignKey(
        Property,
        on_delete=models.CASCADE,
        related_name="reviews",
        blank=True,
        null=True,
    )

    # User review — one user reviews another (landlord reviews tenant, or vice versa)
    reviewed_user = models.ForeignKey(
        Users,
        on_delete=models.CASCADE,
        related_name="reviews_received",
        blank=True,
        null=True,
    )

    booking = models.OneToOneField(
        Booking,
        on_delete=models.CASCADE,
        related_name="review",
        null=True,
        blank=True,
        help_text="Links property review to a specific booking. Null for user-to-user reviews."
    )
    # ── Review Content ────────────────────────────────────────────────────────
    rating  = models.IntegerField(
        validators=[MinValueValidator(1), MaxValueValidator(5)]
    )
    comment = models.TextField(blank=True, null=True)

    # ── Timestamp ─────────────────────────────────────────────────────────────
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        ordering = ["-created_at"]
        # One review per reviewer per target
        # NOTE: Django doesn't support conditional unique_together,
        # so uniqueness per target type is enforced in the serializer.
        verbose_name_plural = "Reviews"

    def __str__(self):
        target = self.property or self.reviewed_user
        return f"{self.reviewer.username} → {target} ({self.rating}★)"

```

# ────────────────────────End Model──────────────────────────────────────



## favorite app ##


# ────────────────────────Start URL─────────────────────────────────────

```
"""Favorites API URL configuration."""
from django.urls import path
from api.favorites_api.views import FavoritesListView, FavoriteDeleteView

urlpatterns = [
    # ── Student: View shortlist + heart a property ────────────────────────────
    path("", FavoritesListView.as_view(), name="favorites-list"),

    # ── Student: Unheart a property (by property id, not favorite id) ─────────
    path("<int:property_id>/", FavoriteDeleteView.as_view(), name="favorite-delete"),
]

        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```
"""Favorites API serializers."""
from django.db import IntegrityError
from rest_framework import serializers
from favorites.models import Favorite
from api.properties_api.serializers import PropertyListSerializer


class FavoriteSerializer(serializers.ModelSerializer):
    property_detail = PropertyListSerializer(source="property", read_only=True)

    class Meta:
        model = Favorite
        fields = ["id", "property", "property_detail", "created_at"]
        read_only_fields = ["id", "created_at"]


class FavoriteCreateSerializer(serializers.ModelSerializer):
    class Meta:
        model = Favorite
        fields = ["property"]

    def validate(self, data):
        user = self.context["request"].user
        prop = data.get("property")

        # 1. Prevent favoriting rented/unavailable properties
        if prop.status != "available":
            raise serializers.ValidationError("You can only save available properties.")

        # 2. Prevent duplicates
        if Favorite.objects.filter(user=user, property=prop).exists():
            raise serializers.ValidationError("You have already saved this property.")

        return data

    def create(self, validated_data, **extra_kwargs):
        """
        extra_kwargs captures the `user=request.user` passed from the view.
        IntegrityError catches rare race conditions (e.g., double-clicking the heart button).
        """
        try:
            return Favorite.objects.create(**validated_data, **extra_kwargs)
        except IntegrityError:
            raise serializers.ValidationError("You have already saved this property.")

```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```
"""Favorites API views."""
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status

from favorites.models import Favorite
from api.favorites_api.serializers import FavoriteSerializer, FavoriteCreateSerializer
from api.accounts_api.permissions import IsStudent


class FavoritesListView(APIView):
    permission_classes = [IsStudent]

    def get(self, request):
        queryset = (
            Favorite.objects
            .filter(user=request.user)
            .select_related("property", "property__landlord")  
            .prefetch_related("property__images")
        )
        serializer = FavoriteSerializer(queryset, many=True, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)

    def post(self, request):
        serializer = FavoriteCreateSerializer(data=request.data, context={"request": request})
        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        # Inject the authenticated student
        favorite = serializer.save(user=request.user)
        return Response(
            FavoriteSerializer(favorite, context={"request": request}).data,
            status=status.HTTP_201_CREATED,
        )


class FavoriteDeleteView(APIView):
    permission_classes = [IsStudent]

    def delete(self, request, property_id):
        try:
            favorite = Favorite.objects.get(user=request.user, property_id=property_id)
        except Favorite.DoesNotExist:
            return Response({"error": "Property not in your shortlist."}, status=status.HTTP_404_NOT_FOUND)

        favorite.delete()
        return Response(status=status.HTTP_204_NO_CONTENT)

```

# ────────────────────────End View──────────────────────────────────────





# ────────────────────────Start Signals────────────────────────────────────
```


```
# ────────────────────────End Signals──────────────────────────────────────





# ────────────────────────Start Apps────────────────────────────────────
```


```
# ────────────────────────End Apps──────────────────────────────────────






# ────────────────────────Start Model────────────────────────────────────

```
"""
Favorites app models.
Handles the student shortlist — the heart button on property cards.

Models:
    - Favorite → a saved property belonging to a student
"""

from django.db import models
from accounts.models import Users
from properties.models import Property


class Favorite(models.Model):
    """
    Represents a property saved to a student's shortlist.

    unique_together prevents the same student from hearting the same
    property twice — the API treats a duplicate POST as a 400 error.
    """

    # ── Parties ───────────────────────────────────────────────────────────────
    user = models.ForeignKey(
        Users,
        on_delete=models.CASCADE,
        related_name="favorites",
        limit_choices_to={"role": "student"},   # only students can shortlist
    )
    property = models.ForeignKey(
        Property,
        on_delete=models.CASCADE,
        related_name="favorited_by",
    )

    # ── Timestamp ─────────────────────────────────────────────────────────────
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        ordering = ["-created_at"]
        # One heart per student per property — enforced at DB level
        unique_together = ("user", "property")

    def __str__(self):
        return f"{self.user.username} ♥ {self.property.title}"
```

# ────────────────────────End Model──────────────────────────────────────



## community app ##


# ────────────────────────Start URL─────────────────────────────────────

```
"""Community API URL configuration."""

from django.urls import path
from api.community_api.views import (
    GroupListView,
    MyGroupsView,
    GroupDetailView,
    GroupCreateView,
    GroupJoinView,
    GroupLeaveView,
    PostListView,
    PostCreateView,
)

urlpatterns = [
    # ── Groups ───────────────────────────────────────────────────────────────
    path("groups/",              GroupListView.as_view(),   name="community-group-list"),
    path("groups/my/",           MyGroupsView.as_view(),    name="community-my-groups"),
    path("groups/<int:group_id>/",       GroupDetailView.as_view(), name="community-group-detail"),
    path("groups/create/",       GroupCreateView.as_view(), name="community-group-create"),
    path("groups/<int:group_id>/join/",  GroupJoinView.as_view(),   name="community-group-join"),
    path("groups/<int:group_id>/leave/", GroupLeaveView.as_view(),  name="community-group-leave"),

    # ── Posts ─────────────────────────────────────────────────────────────────
    path("posts/",               PostListView.as_view(),    name="community-post-list"),
    path("posts/create/",        PostCreateView.as_view(),  name="community-post-create"),
]

        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```
"""
Community API serializers.

GroupSerializer          — full group detail (used in detail + my-groups)
GroupListSerializer      — lightweight card for browse list
GroupCreateSerializer    — POST create group
PostSerializer           — full post response
PostCreateSerializer     — POST new post
"""

from rest_framework import serializers
from community.models import Group, GroupMembership, Post


# ── Shared author snippet ─────────────────────────────────────────────────────

class _AuthorSerializer(serializers.Serializer):
    """Minimal author info embedded in Post responses."""
    id         = serializers.IntegerField(read_only=True)
    first_name = serializers.CharField(read_only=True)
    last_name  = serializers.CharField(read_only=True)


# ── Group serializers ─────────────────────────────────────────────────────────

class GroupListSerializer(serializers.ModelSerializer):
    """
    Lightweight serializer for the group browse list.
    Adds `is_member` so the frontend can show Join / Leave immediately.
    """
    is_member = serializers.BooleanField(read_only=True)

    class Meta:
        model  = Group
        fields = [
            "id", "name", "category", "cover_image",
            "member_count", "is_private", "is_member", "created_at",
        ]

    


class GroupSerializer(serializers.ModelSerializer):
    """
    Full group detail — used for the group detail page and my-groups list.
    Includes creator name and the requesting user's membership role.
    """
    is_member   = serializers.SerializerMethodField()
    member_role = serializers.SerializerMethodField()
    creator_name = serializers.SerializerMethodField()

    class Meta:
        model  = Group
        fields = [
            "id", "name", "description", "category", "cover_image",
            "member_count", "is_private",
            "creator_name", "is_member", "member_role",
            "created_at", "updated_at",
        ]

    def get_creator_name(self, obj):
        if obj.creator:
            return f"{obj.creator.first_name} {obj.creator.last_name}".strip()
        return None

    def get_is_member(self, obj):
        request = self.context.get("request")
        if not request or not request.user.is_authenticated:
            return False
        return obj.memberships.filter(user=request.user).exists()

    def get_member_role(self, obj):
        """Returns 'admin', 'member', or None if not a member."""
        request = self.context.get("request")
        if not request or not request.user.is_authenticated:
            return None
        membership = obj.memberships.filter(user=request.user).first()
        return membership.role if membership else None


class GroupCreateSerializer(serializers.ModelSerializer):
    """Accepts name, description, category, cover_image, is_private."""

    class Meta:
        model  = Group
        fields = ["name", "description", "category", "cover_image", "is_private"]

    def validate_name(self, value):
        if Group.objects.filter(name__iexact=value).exists():
            raise serializers.ValidationError("A group with this name already exists.")
        return value

    def create(self, validated_data):
        """
        Create the group and automatically enrol the creator as admin.
        member_count signal will fire and set count to 1.
        """
        creator = validated_data.pop("creator")
        group   = Group.objects.create(creator=creator, **validated_data)
        GroupMembership.objects.create(user=creator, group=group, role="admin")
        return group


# ── Post serializers ──────────────────────────────────────────────────────────

class PostSerializer(serializers.ModelSerializer):
    """Full post response with nested author info."""
    author = _AuthorSerializer(read_only=True)

    class Meta:
        model  = Post
        fields = ["id", "author", "group", "content", "image", "created_at", "updated_at"]


class PostCreateSerializer(serializers.ModelSerializer):
    """
    Accepts content and optional image.
    author and group are injected server-side — never trusted from request body.
    """

    class Meta:
        model  = Post
        fields = ["content", "image"]

    def validate_content(self, value):
        if not value.strip():
            raise serializers.ValidationError("Post content cannot be empty.")
        return value

    def create(self, validated_data):
        """author and group are passed via save(author=..., group=...)."""
        return Post.objects.create(**validated_data)


```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```
"""
Community API views.

GroupListView          GET  /api/community/groups/
MyGroupsView           GET  /api/community/groups/my/
GroupDetailView        GET  /api/community/groups/<id>/
GroupCreateView        POST /api/community/groups/create/
GroupJoinView          POST /api/community/groups/<id>/join/
GroupLeaveView         POST /api/community/groups/<id>/leave/
PostListView           GET  /api/community/posts/?group=<id>
PostCreateView         POST /api/community/posts/
"""
from django.db.models import Exists, OuterRef 
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from rest_framework.permissions import IsAuthenticated

from community.models import Group, GroupMembership, Post
from api.community_api.serializers import (
    GroupListSerializer,
    GroupSerializer,
    GroupCreateSerializer,
    PostSerializer,
    PostCreateSerializer,
)


class GroupListView(APIView):
    """
    GET /api/community/groups/
    Returns all groups as lightweight cards.
    Optional filter: ?category=university|housing|social|study|other

    Open to all authenticated users (students + landlords can browse).
    """
    permission_classes = [IsAuthenticated]

    def get(self, request):
        queryset = Group.objects.all()

        category = request.query_params.get("category")
        if category:
            queryset = queryset.filter(category=category)

        queryset = queryset.annotate(
            is_member=Exists(
                GroupMembership.objects.filter(group=OuterRef('pk'), user=request.user)
            )
        )

        serializer = GroupListSerializer(queryset, many=True, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)


class MyGroupsView(APIView):
    """
    GET /api/community/groups/my/
    Returns all groups the requesting user has joined.
    """
    permission_classes = [IsAuthenticated]

    def get(self, request):
        group_ids = GroupMembership.objects.filter(user=request.user).values_list("group_id", flat=True)
        queryset  = Group.objects.filter(id__in=group_ids)
        serializer = GroupSerializer(queryset, many=True, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)


class GroupDetailView(APIView):
    """
    GET /api/community/groups/<id>/
    Returns full group detail including membership status of the requester.
    """
    permission_classes = [IsAuthenticated]

    def get(self, request, group_id):
        try:
            group = Group.objects.get(id=group_id)
        except Group.DoesNotExist:
            return Response({"error": "Group not found."}, status=status.HTTP_404_NOT_FOUND)

        serializer = GroupSerializer(group, context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)


class GroupCreateView(APIView):
    """
    POST /api/community/groups/create/
    Creates a new group and auto-enrolls the creator as admin.
    Any authenticated user can create a group.
    """
    permission_classes = [IsAuthenticated]

    def post(self, request):
        serializer = GroupCreateSerializer(data=request.data)
        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        group = serializer.save(creator=request.user)
        return Response(
            GroupSerializer(group, context={"request": request}).data,
            status=status.HTTP_201_CREATED,
        )


class GroupJoinView(APIView):
    """
    POST /api/community/groups/<id>/join/
    Join a group.  Returns 400 if already a member.
    """
    permission_classes = [IsAuthenticated]

    def post(self, request, group_id):
        try:
            group = Group.objects.get(id=group_id)
        except Group.DoesNotExist:
            return Response({"error": "Group not found."}, status=status.HTTP_404_NOT_FOUND)

        if GroupMembership.objects.filter(user=request.user, group=group).exists():
            return Response({"error": "You are already a member of this group."}, status=status.HTTP_400_BAD_REQUEST)

        GroupMembership.objects.create(user=request.user, group=group, role="member")
        return Response(
            GroupSerializer(group, context={"request": request}).data,
            status=status.HTTP_200_OK,
        )


class GroupLeaveView(APIView):
    """
    POST /api/community/groups/<id>/leave/
    Leave a group.
    - Returns 400 if the user is not a member.
    - Prevents the last admin from leaving (they must first promote another member).
    """
    permission_classes = [IsAuthenticated]

    def post(self, request, group_id):
        try:
            group = Group.objects.get(id=group_id)
        except Group.DoesNotExist:
            return Response({"error": "Group not found."}, status=status.HTTP_404_NOT_FOUND)

        try:
            membership = GroupMembership.objects.get(user=request.user, group=group)
        except GroupMembership.DoesNotExist:
            return Response({"error": "You are not a member of this group."}, status=status.HTTP_400_BAD_REQUEST)

        # Guard: don't let the last admin abandon the group
        if membership.role == "admin":
            other_admins = GroupMembership.objects.filter(group=group, role="admin").exclude(user=request.user)
            if not other_admins.exists():
                return Response(
                    {"error": "You are the only admin. Promote another member before leaving."},
                    status=status.HTTP_400_BAD_REQUEST,
                )

        membership.delete()
        return Response({"detail": "You have left the group."}, status=status.HTTP_200_OK)


class PostListView(APIView):
    """
    GET /api/community/posts/?group=<id>
    Returns all posts for a group, newest first.
    The requesting user must be a member of the group to view its posts.
    """
    permission_classes = [IsAuthenticated]

    def get(self, request):
        group_id = request.query_params.get("group")
        if not group_id:
            return Response({"error": "Query param 'group' is required."}, status=status.HTTP_400_BAD_REQUEST)

        try:
            group = Group.objects.get(id=group_id)
        except Group.DoesNotExist:
            return Response({"error": "Group not found."}, status=status.HTTP_404_NOT_FOUND)

        # Only members can read posts
        if not GroupMembership.objects.filter(user=request.user, group=group).exists():
            return Response(
                {"error": "You must join this group to view its posts."},
                status=status.HTTP_403_FORBIDDEN,
            )

        posts = Post.objects.filter(group=group).select_related("author")
        serializer = PostSerializer(posts, many=True,context={"request": request})
        return Response(serializer.data, status=status.HTTP_200_OK)


class PostCreateView(APIView):
    """
    POST /api/community/posts/
    Body: { "group": <id>, "content": "...", "image": <optional file> }

    The user must be a member of the group to post.
    author is always the requesting user — never trusted from request body.
    """
    permission_classes = [IsAuthenticated]

    def post(self, request):
        group_id = request.data.get("group")
        if not group_id:
            return Response({"error": "Field 'group' is required."}, status=status.HTTP_400_BAD_REQUEST)

        try:
            group = Group.objects.get(id=group_id)
        except Group.DoesNotExist:
            return Response({"error": "Group not found."}, status=status.HTTP_404_NOT_FOUND)

        # Must be a member to post
        if not GroupMembership.objects.filter(user=request.user, group=group).exists():
            return Response(
                {"error": "You must join this group before posting."},
                status=status.HTTP_403_FORBIDDEN,
            )

        serializer = PostCreateSerializer(data=request.data)
        if not serializer.is_valid():
            return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)

        post = serializer.save(author=request.user, group=group)
        return Response(
            PostSerializer(post,context={"request": request}).data
                        , status=status.HTTP_201_CREATED
                        )


```

# ────────────────────────End View──────────────────────────────────────





# ────────────────────────Start Signals────────────────────────────────────
```
"""
Signals for the community app.

Keeps Group.member_count in sync whenever a GroupMembership is
created or deleted, avoiding expensive COUNT(*) queries on every
group list request.
"""

from django.db.models.signals import post_save, post_delete
from django.dispatch import receiver

from community.models import GroupMembership


@receiver(post_save, sender=GroupMembership)
def increment_member_count(sender, instance, created, **kwargs):
    """Bump member_count when a new membership row is created."""
    if created:
        # Use F() expression to avoid race conditions
        from django.db.models import F
        instance.group.__class__.objects.filter(pk=instance.group_id).update(
            member_count=F("member_count") + 1
        )


@receiver(post_delete, sender=GroupMembership)
def decrement_member_count(sender, instance, **kwargs):
    """Lower member_count when a membership is removed."""
    from django.db.models import F
    instance.group.__class__.objects.filter(pk=instance.group_id).update(
        member_count=F("member_count") - 1
    )


```
# ────────────────────────End Signals──────────────────────────────────────





# ────────────────────────Start Apps────────────────────────────────────
```
"""Community app config — wires signals on startup."""

from django.apps import AppConfig


class CommunityConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name = "community"

    def ready(self):
        import community.signals  # noqa: F401 — registers signal handlers

```
# ────────────────────────End Apps──────────────────────────────────────








# ────────────────────────Start Model────────────────────────────────────

```
"""
Community models: Group, GroupMembership, Post.

Groups are student-created spaces (e.g. "Cairo University 2025", "Engineers Only").
Any authenticated user can browse groups; students join/post.
"""

from django.db import models
from django.conf import settings


class Group(models.Model):
    """A community group that students can join and post in."""

    CATEGORY_CHOICES = [
        ("university", "University"),
        ("housing",    "Housing"),
        ("social",     "Social"),
        ("study",      "Study"),
        ("other",      "Other"),
    ]

    creator     = models.ForeignKey(
        settings.AUTH_USER_MODEL,
        on_delete=models.SET_NULL,
        null=True,
        related_name="created_groups",
    )
    name        = models.CharField(max_length=120, unique=True)
    description = models.TextField(blank=True)
    category    = models.CharField(max_length=20, choices=CATEGORY_CHOICES, default="other")
    cover_image = models.ImageField(upload_to="group_covers/", null=True, blank=True)

    # Denormalised counter — updated by signals to avoid COUNT(*) on every list request
    member_count = models.PositiveIntegerField(default=0)

    is_private  = models.BooleanField(
        default=False,
        help_text="Private groups require approval to join (reserved for future use).",
    )

    created_at  = models.DateTimeField(auto_now_add=True)
    updated_at  = models.DateTimeField(auto_now=True)

    class Meta:
        ordering = ["-created_at"]

    def __str__(self):
        return self.name


class GroupMembership(models.Model):
    """
    Through-model between Users and Group.
    One row = one member.  unique_together prevents duplicate joins.
    """

    ROLE_CHOICES = [
        ("member", "Member"),
        ("admin",  "Admin"),
    ]

    user       = models.ForeignKey(
        settings.AUTH_USER_MODEL,
        on_delete=models.CASCADE,
        related_name="group_memberships",
    )
    group      = models.ForeignKey(
        Group,
        on_delete=models.CASCADE,
        related_name="memberships",
    )
    role       = models.CharField(max_length=10, choices=ROLE_CHOICES, default="member")
    joined_at  = models.DateTimeField(auto_now_add=True)

    class Meta:
        unique_together = ("user", "group")
        ordering        = ["joined_at"]

    def __str__(self):
        return f"{self.user} → {self.group} ({self.role})"


class Post(models.Model):
    """A post inside a Group."""

    author  = models.ForeignKey(
        settings.AUTH_USER_MODEL,
        on_delete=models.CASCADE,
        related_name="community_posts",
    )
    group   = models.ForeignKey(
        Group,
        on_delete=models.CASCADE,
        related_name="posts",
    )
    content = models.TextField()
    image   = models.ImageField(upload_to="post_images/", null=True, blank=True)

    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    class Meta:
        ordering = ["-created_at"]

    def __str__(self):
        return f"Post by {self.author} in {self.group} @ {self.created_at:%Y-%m-%d}"

```

# ────────────────────────End Model──────────────────────────────────────



## notifications app ##


# ────────────────────────Start URL─────────────────────────────────────

```
"""Notifications API URL configuration."""

from django.urls import path
from api.notifications_api.views import (
    NotificationListView,
    MarkAllReadView,
    MarkOneReadView,
)

urlpatterns = [
    # ── Bell feed ─────────────────────────────────────────────────────────────
    path("",                    NotificationListView.as_view(), name="notifications-list"),

    # ── Mark all read ─────────────────────────────────────────────────────────
    path("read/",               MarkAllReadView.as_view(),      name="notifications-read-all"),

    # ── Mark one read ─────────────────────────────────────────────────────────
    path("<int:notification_id>/read/", MarkOneReadView.as_view(), name="notifications-read-one"),
]

        
```

# ────────────────────────End URL───────────────────────────────────────



# ────────────────────────Start Serializer──────────────────────────────

```
"""
Notifications API serializers.

Serializers:
    - NotificationSerializer → read serializer for the bell feed
"""

from rest_framework import serializers
from notifications.models import Notification


class NotificationSerializer(serializers.ModelSerializer):
    """
    Read serializer for a single notification item.
    actor_name is included so the frontend can show "Ahmed sent you a request"
    without a second profile request.
    """

    actor_name   = serializers.SerializerMethodField()
    actor_avatar = serializers.SerializerMethodField()

    class Meta:
        model  = Notification
        fields = [
            "id",
            "notification_type",
            "title",
            "message",
            "data",
            "is_read",
            "actor_name",
            "actor_avatar",
            "created_at",
        ]
        read_only_fields = fields  # this serializer is read-only

    def get_actor_name(self, obj):
        """Returns the actor's full display name, or None for system notifications."""
        if obj.actor is None:
            return None
        full = f"{obj.actor.first_name or ''} {obj.actor.last_name or ''}".strip()
        return full or obj.actor.username

    def get_actor_avatar(self, obj):
        """Returns the actor's profile picture URL, or None."""
        if obj.actor is None or not obj.actor.profile_picture:
            return None
        request = self.context.get("request")
        if request:
            return request.build_absolute_uri(obj.actor.profile_picture.url)
        return obj.actor.profile_picture.url

```

# ────────────────────────End Serializer────────────────────────────────




# ────────────────────────Start View────────────────────────────────────

```
"""
Notifications API views.

Views:
    - NotificationListView   → GET  /api/notifications/         ← bell feed
    - MarkAllReadView        → POST /api/notifications/read/    ← mark all as read
    - MarkOneReadView        → PATCH /api/notifications/<id>/read/ ← mark one as read
"""

from rest_framework.views    import APIView
from rest_framework.response import Response
from rest_framework          import status
from rest_framework.permissions import IsAuthenticated

from notifications.models import Notification
from api.notifications_api.serializers import NotificationSerializer


class NotificationListView(APIView):
    """
    GET /api/notifications/
    Returns the requesting user's full notification feed, newest first.
    Also returns unread_count so the bell badge renders without parsing the list.

    Optional query param: ?unread=true → returns only unread notifications.
    """
    permission_classes = [IsAuthenticated]

    def get(self, request):
        qs = Notification.objects.filter(recipient=request.user)
        unread_count = qs.filter(is_read=False).count()

        if request.query_params.get("unread") == "true":
            qs = qs.filter(is_read=False)

        serializer = NotificationSerializer(qs, many=True, context={"request": request})
        return Response(
            {
                "unread_count":    unread_count,
                "notifications":   serializer.data,
            },
            status=status.HTTP_200_OK,
        )


class MarkAllReadView(APIView):
    """
    POST /api/notifications/read/
    Bulk-marks every unread notification for the requesting user as read.
    Body: (none required)
    """
    permission_classes = [IsAuthenticated]

    def post(self, request):
        updated = (
            Notification.objects
            .filter(recipient=request.user, is_read=False)
            .update(is_read=True)
        )
        return Response(
            {"marked_read": updated},
            status=status.HTTP_200_OK,
        )


class MarkOneReadView(APIView):
    """
    PATCH /api/notifications/<id>/read/
    Marks a single notification as read.
    Returns 404 if the notification doesn't belong to the requesting user.
    """
    permission_classes = [IsAuthenticated]

    def patch(self, request, notification_id):
        try:
            notification = Notification.objects.get(
                id=notification_id,
                recipient=request.user,
            )
        except Notification.DoesNotExist:
            return Response(
                {"error": "Notification not found."},
                status=status.HTTP_404_NOT_FOUND,
            )

        notification.is_read = True
        notification.save(update_fields=["is_read"])

        return Response(
            NotificationSerializer(notification, context={"request": request}).data,
            status=status.HTTP_200_OK,
        )

```

# ────────────────────────End View──────────────────────────────────────



# ────────────────────────Start Services────────────────────────────────────
```
"""
Notifications app services.
Single entry point for creating notifications.

Import this function in any signal or view that needs to fire a notification:

    from notifications.services import push_notification
"""

from notifications.models import Notification


def push_notification(recipient, notification_type, title, message, actor=None, data=None):
    """
    Create and persist a single notification.

    Args:
        recipient         (Users)  : the user who will see this in their bell feed
        notification_type (str)   : one of Notification.TYPE_CHOICES keys
        title             (str)   : short heading shown in the notification card
        message           (str)   : one-sentence body text
        actor             (Users) : user who caused the event (None for system notices)
        data              (dict)  : optional extra payload for frontend deep-linking

    Returns:
        Notification instance
    """
    return Notification.objects.create(
        recipient=recipient,
        actor=actor,
        notification_type=notification_type,
        title=title,
        message=message,
        data=data or {},
    )

```
# ────────────────────────End Services────────────────────────────────────


# ────────────────────────Start Signals────────────────────────────────────
```

"""
notifications/signals.py
"""

from django.db.models.signals import post_save
from django.dispatch import receiver
from asgiref.sync import async_to_sync
from channels.layers import get_channel_layer

from accounts.models  import Users
from bookings.models  import Booking
from reviews.models   import Review
from messaging.models import Message

from notifications.services import push_notification


def _broadcast(notification):
    """
    After saving a notification to DB, push it live to the user's
    personal WebSocket group: notifications_<user_id>
    """
    channel_layer = get_channel_layer()
    if channel_layer is None:
        return

    async_to_sync(channel_layer.group_send)(
        f"notifications_{notification.recipient.id}",
        {
            "type":              "notify",           # maps to NotificationConsumer.notify()
            "id":                notification.id,
            "notification_type": notification.notification_type,
            "title":             notification.title,
            "message":           notification.message,
            "data":              notification.data,
            "is_read":           notification.is_read,
            "created_at":        str(notification.created_at),
        },
    )


# ── Accounts: welcome ─────────────────────────────────────────────────────────

@receiver(post_save, sender=Users)
def welcome_notification(sender, instance, created, **kwargs):
    if not created:
        return
    n = push_notification(
        recipient=instance,
        notification_type="welcome",
        title="Welcome to StudentHub! 🎉",
        message="Your account is ready. Start exploring rooms and finding roommates.",
    )
    _broadcast(n)


# ── Bookings ──────────────────────────────────────────────────────────────────

@receiver(post_save, sender=Booking)
def booking_notification(sender, instance, created, **kwargs):
    property_title = instance.property.title

    if created:
        n = push_notification(
            recipient=instance.property.landlord,          # ← was .owner (bug fixed)
            actor=instance.tenant,
            notification_type="booking_request",
            title="New Booking Request",
            message=f"{instance.tenant.username} wants to book '{property_title}'.",
            data={
                "booking_id":     instance.id,
                "property_id":    instance.property.id,
                "property_title": property_title,
            },
        )
        _broadcast(n)
        return

    STATUS_MESSAGES = {
        "deposit_paid": ("Deposit Received 💰",   f"Deposit paid for '{property_title}'. Please confirm the booking."),
        "confirmed":    ("Booking Confirmed ✅",   f"Your booking for '{property_title}' has been confirmed."),
        "completed":    ("Stay Completed 🏁",      f"Your stay at '{property_title}' has been marked as completed."),
        "cancelled":    ("Booking Cancelled",      f"Your booking for '{property_title}' has been cancelled."),
    }

    if instance.status not in STATUS_MESSAGES:
        return

    title, message = STATUS_MESSAGES[instance.status]

    # deposit_paid → landlord needs to act; everything else → tenant is informed
    recipient = (
        instance.property.landlord
        if instance.status == "deposit_paid"
        else instance.tenant
    )

    n = push_notification(
        recipient=recipient,
        actor=instance.property.landlord,
        notification_type="booking_update",
        title=title,
        message=message,
        data={
            "booking_id":     instance.id,
            "property_id":    instance.property.id,
            "property_title": property_title,
            "new_status":     instance.status,
        },
    )
    _broadcast(n)


# ── Reviews ───────────────────────────────────────────────────────────────────

@receiver(post_save, sender=Review)
def review_notification(sender, instance, created, **kwargs):
    if not created:
        return

    if instance.property:
        recipient = instance.property.landlord          # ← was .owner (bug fixed)
        subject   = f"your property '{instance.property.title}'"
        data      = {"property_id": instance.property.id}
    elif instance.reviewed_user:
        recipient = instance.reviewed_user
        subject   = "your profile"
        data      = {"reviewer_id": instance.reviewer.id}
    else:
        return

    n = push_notification(
        recipient=recipient,
        actor=instance.reviewer,
        notification_type="new_review",
        title="New Review Posted ⭐",
        message=f"{instance.reviewer.username} left a {instance.rating}-star review on {subject}.",
        data=data,
    )
    _broadcast(n)


# ── Messaging ─────────────────────────────────────────────────────────────────

@receiver(post_save, sender=Message)
def message_notification(sender, instance, created, **kwargs):
    if not created:
        return

    # ← was instance.conversation.participants (bug fixed — use initiator/receiver)
    conv = instance.conversation
    recipients = [conv.initiator, conv.receiver]

    for user in recipients:
        if user == instance.sender:
            continue
        n = push_notification(
            recipient=user,
            actor=instance.sender,
            notification_type="new_message",
            title="New Message 💬",
            message=f"{instance.sender.username} sent you a message.",
            data={
                "conversation_id": conv.id,
                "message_id":      instance.id,
            },
        )
        _broadcast(n)

```
# ────────────────────────End Signals──────────────────────────────────────





# ────────────────────────Start Apps────────────────────────────────────
```

"""Notifications app config — wires signals on startup."""

from django.apps import AppConfig


class NotificationsConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name = "notifications"

    def ready(self):
        import notifications.signals  # noqa: F401 — registers all signal handlers
```
# ────────────────────────End Apps──────────────────────────────────────






# ────────────────────────Start Model────────────────────────────────────

```
"""
Notifications app models.
Stores the bell icon feed for every user.

Models:
    - Notification → a single notification item for one recipient
"""

from django.db import models
from accounts.models import Users


class Notification(models.Model):
    """
    One notification entry in a user's bell feed.

    actor   → who triggered it (e.g. the student who sent a roommate request)
    target  → always the person receiving the notification
    type    → determines the icon, copy, and deep-link the frontend renders
    data    → optional JSON payload for dynamic content (e.g. property title, booking id)
    """

    TYPE_CHOICES = [
        ("welcome",          "Welcome"),
        ("booking_request",  "Booking Request"),
        ("booking_update",   "Booking Update"),
        ("roommate_request", "Roommate Request"),
        ("roommate_update",  "Roommate Update"),
        ("new_message",      "New Message"),
        ("new_review",       "New Review"),
        ("rating_update",    "Rating Update"),
        ("payment",          "Payment"),
        ("system",           "System"),
    ]

    # ── Recipients & actors ───────────────────────────────────────────────────
    recipient = models.ForeignKey(
        Users,
        on_delete=models.CASCADE,
        related_name="notifications",
    )
    actor = models.ForeignKey(
        Users,
        on_delete=models.SET_NULL,
        null=True,
        blank=True,
        related_name="triggered_notifications",
        # NULL for system / welcome notifications that have no actor
    )

    # ── Content ───────────────────────────────────────────────────────────────
    notification_type = models.CharField(max_length=30, choices=TYPE_CHOICES)
    title   = models.CharField(max_length=255)
    message = models.TextField()

    # Optional JSON bag: e.g. {"booking_id": 7, "property_title": "Studio in Maadi"}
    data = models.JSONField(default=dict, blank=True)

    # ── State ─────────────────────────────────────────────────────────────────
    is_read    = models.BooleanField(default=False)
    created_at = models.DateTimeField(auto_now_add=True)

    class Meta:
        ordering = ["-created_at"]  # newest first in every queryset

    def __str__(self):
        return f"[{self.notification_type}] → {self.recipient.username}: {self.title}"
```

# ────────────────────────End Model──────────────────────────────────────



## notifications Websocket setup ##


# ────────────────────────Start Routing────────────────────────────────────
```
"""
notifications/routing.py
ws://localhost:8000/ws/notifications/?token=<jwt>
"""

from django.urls import re_path
from notifications.consumers import NotificationConsumer

websocket_urlpatterns = [
    re_path(r"^ws/notifications/$", NotificationConsumer.as_asgi()),
]
```
# ────────────────────────End Routing──────────────────────────────────────

# ────────────────────────Start Middleware────────────────────────────────────
```
from urllib.parse import parse_qs


from channels.middleware import BaseMiddleware

from channels.db import database_sync_to_async
from django.contrib.auth.models import AnonymousUser
from rest_framework_simplejwt.tokens import AccessToken
from rest_framework_simplejwt.exceptions import InvalidToken , TokenError

from accounts.models import Users

@database_sync_to_async
def get_user_from_token(token_key):

    """
    Validates the JWT access token and returns the matching User.
    Returns AnonymousUser if the token is missing, invalid, or expired.
    """
    try:
        token = AccessToken(token_key)
        return Users.objects.get(id=token["user_id"])
    except (InvalidToken, TokenError, Users.DoesNotExist):
        return AnonymousUser()


class JWTAuthMiddleware(BaseMiddleware):

    """
    Reads the JWT token from the WebSocket query string.

    The browser connects like:
        ws://localhost:8000/ws/chat/5/?token=eyJ...

    Normal DRF authentication (Authorization header) does not work for
    WebSocket connections — headers are not accessible the same way.
    Passing the token as a query param is the standard workaround.
    """
    async def __call__(self, scope, receive, send):
        query_string = scope.get("query_string", b"").decode()
        params       = parse_qs(query_string)
        token_list   = params.get("token",[])
        token        = token_list[0] if token_list else None

        scope["user"] = (
            await get_user_from_token(token)
            if token
            else AnonymousUser()
        )

        return await super().__call__(scope, receive, send)
```
# ────────────────────────End Middleware──────────────────────────────────────





# ────────────────────────Start Consumers────────────────────────────────────
```
"""
notifications/consumers.py

One-way personal channel — server pushes, client only connects/disconnects.
Connection URL:  ws://localhost:8000/ws/notifications/?token=<jwt>
Group name:      notifications_<user_id>
"""

import json
from channels.generic.websocket import AsyncWebsocketConsumer
from django.contrib.auth.models import AnonymousUser


class NotificationConsumer(AsyncWebsocketConsumer):
    """
    Much simpler than ChatConsumer — no receive() logic needed.
    The only job is:
        connect()    → authenticate → join personal group → accept
        notify()     → forward group_send event to this WebSocket client
        disconnect() → leave group
    """

    async def connect(self):
        self.user = self.scope["user"]

        if isinstance(self.user, AnonymousUser):
            await self.close()
            return

        # Personal group — only this user's notifications land here
        self.group_name = f"notifications_{self.user.id}"

        await self.channel_layer.group_add(self.group_name, self.channel_name)
        await self.accept()

    async def disconnect(self, close_code):
        if hasattr(self, "group_name"):
            await self.channel_layer.group_discard(self.group_name, self.channel_name)

    # ── Event handler ─────────────────────────────────────────────────────────
    # Called by channel layer when _broadcast() does group_send(type="notify")

    async def notify(self, event):
        """Push the notification payload to the connected WebSocket client."""
        await self.send(text_data=json.dumps(event))
```
# ────────────────────────End Consumers──────────────────────────────────────





# ────────────────────────Start Asgi────────────────────────────────────
```
"""
ASGI entry point for StudentHub.

Replaces the default wsgi.py as the server gateway when running with
Daphne or Uvicorn. Routes incoming connections by protocol:

    HTTP      → Django URL router → DRF views  (same as before)
    WebSocket → JWTAuthMiddleware → ChatConsumer

wsgi.py can stay in the project — it is still used by some deployment
tools for the HTTP-only path.
"""

import os
from django.core.asgi import get_asgi_application

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")

# get_asgi_application() must be called before any models or consumers
# are imported so Django's app registry is fully loaded first.
django_asgi_app = get_asgi_application()

from channels.routing import ProtocolTypeRouter, URLRouter
from channels.security.websocket import AllowedHostsOriginValidator
from messaging.middleware import JWTAuthMiddleware
from messaging.routing      import websocket_urlpatterns as chat_patterns
from notifications.routing  import websocket_urlpatterns as notification_patterns

application = ProtocolTypeRouter({
    # All normal HTTP traffic goes through Django as usual
    "http": django_asgi_app,

    # WebSocket connections are authenticated first, then routed to consumers
    "websocket":# AllowedHostsOriginValidator(
        JWTAuthMiddleware(
            URLRouter(
                chat_patterns + notification_patterns   
            )
        )
    #),
})

```
# ────────────────────────End Asgi──────────────────────────────────────









# ────────────────────────Start Apps────────────────────────────────────
```
"""Notifications app config — wires signals on startup."""

from django.apps import AppConfig


class NotificationsConfig(AppConfig):
    default_auto_field = "django.db.models.BigAutoField"
    name = "notifications"

    def ready(self):
        import notifications.signals  # noqa: F401 — registers all signal handlers
```
# ────────────────────────End Apps──────────────────────────────────────













